# BigAlpha 2026 All-Blocks XGBoost — Production Submission v1

Upload this generated notebook to the competition platform.

Frozen architecture:
- E4 factor zoo: 66
- Phase 1 microstructure: 25
- Phase 3 selected dynamics: 39
- Total: 130

The submission trains only on fixed public 2021–2023 data, uses raw next-day close-to-close returns, uses injected data sources only for prediction, predicts the final requested trading day without test labels, and returns only `date / instrument / factor`.


In [ ]:
def main(datasources, start_date, end_date):
    """BigAlpha 2026 All-Blocks XGBoost production submission."""
    import gc
    import os
    import warnings

    import numpy as np
    import pandas as pd
    import dai
    import xgboost as xgb

    warnings.filterwarnings("ignore")

    if not isinstance(datasources, dict):
        raise TypeError("datasources must be a dictionary supplied by the platform.")
    missing_sources = [key for key in ("bar1m", "financial") if key not in datasources]
    if missing_sources:
        raise KeyError(f"datasources missing required keys: {missing_sources}")

    BUILD_ID = "BIGALPHA_ALL_BLOCKS_XGB_SUBMISSION_V1_20260805"
    SOURCE_HASHES = {'e4_source': '5cfeeb28f3077305701bb932b65688fbf1002ff4cee6bd488d2f692e3a20b65d', 'phase1_source': 'e66ae7c8e1aa5c4d77929a13867367ce9e75170c726fffb00a5171be1c1b3773', 'phase3_source': '59137e17ddcd6cfe4d577ee2fad5e66304f6abaaf9443235f53a9a5c47007d7b', 'e4_features': '9b57e970bf7dbdd1f2a97f27d1d215f1d6333e92de52a0e3e7460e7744842c03', 'phase1_features': 'ad93714fa2fd8e60975c5356530bd8565fd9f74751d0e9e61053eeffdb8d11d0', 'phase3_features': 'c24929ccdb45980be5c0f5764c072aa420e54bbd949875d190f289819a0089f7'}
    TRAIN_BAR1M_TABLE = "bigalpha_2026_stock_bar1m"
    TRAIN_FINANCIAL_TABLE = "bigalpha_2026_financial"
    INSTRUMENT_TABLE = "bigalpha_2026_instruments"
    TRAIN_START = pd.Timestamp("2021-01-01")
    DEV_END = pd.Timestamp("2022-12-31")
    TRAIN_END = pd.Timestamp("2023-12-31")
    LABEL_BUFFER_DAYS = 14
    RANDOM_STATE = 2026
    MAX_ROWS_PER_DATE = 350
    THREADS = min(16, os.cpu_count() or 4)

    E4_FEATURES = ['x__gtja_070', 'x__current__turn', 'x__wq_016', 'x__wq_013', 'x__gtja_095', 'x__current__volatility_5', 'x__wq_054', 'x__current__atr_pct', 'x__gtja_080', 'x__wq_025', 'x__wq_055', 'x__wq_044', 'x__wq_033', 'x__gtja_043', 'x__wq_041', 'x__current__netflow_amount_rate_main', 'x__wq_018', 'x__gtja_118', 'x__gtja_046', 'x__gtja_003', 'x__current__reversal_5', 'x__wq_034', 'x__gtja_031', 'x__gtja_020', 'x__gtja_040', 'x__wq_003', 'x__gtja_014', 'x__gtja_038', 'x__gtja_015', 'x__gtja_002', 'x__current__D_ROE_4Q', 'x__gtja_112', 'x__current__EP_CHINA', 'x__gtja_008', 'x__current__D_ROA_4Q', 'x__wq_002', 'x__gtja_011', 'x__gtja_110', 'x__gtja_133', 'x__wq_042', 'x__gtja_037', 'x__current__CFOA_LAG', 'x__wq_012', 'x__current__ROE_Q', 'x__current__ROA_Q', 'x__wq_020', 'x__current__OCFP', 'x__current__SP', 'x__current__PROFIT_MARGIN', 'x__current__DPIA', 'x__current__OPA_LAG', 'x__wq_005', 'x__current__ASSET_TURNOVER', 'x__current__NOA', 'x__current__CASH_TO_ASSETS', 'x__current__GPA_TTM', 'x__current__POA', 'x__current__GLA_TTM', 'x__current__RECEIVABLES_GROWTH', 'x__current__OA_CF', 'x__current__INVENTORY_CHANGE_A', 'x__current__gross_profit_rate_ttm', 'x__current__INVENTORY_GROWTH_YOY', 'x__current__NDP', 'x__current__ASSET_GROWTH_YOY', 'x__current__current_ratio_lf']
    PHASE1_FEATURES = ['depth_per_spread__std', 'obi_1__last_z', 'linear_pressure__mean', 'abs_mid_return__std', 'absorption_score__mean', 'fragility__std', 'signed_pressure__last_z', 'trade_to_depth__std', 'fragility__close30_minus_full', 'alignment_obi_return__mean', 'official_pressure__std', 'log_depth_5__close30_minus_full', 'fragility__last_z', 'obi_5__std', 'log_depth_1__std', 'log_depth_5__last_z', 'log_depth_5__std', 'top_depth_share__std', 'absorption_score__last_z', 'official_pressure__last_z', 'near_far_obi__std', 'obi_1__std', 'weighted_imbalance__last_z', 'obi_1__close30_minus_full', 'response_ratio__mean']
    PHASE3_FEATURES = ['mlofi__close30_mean', 'mlofi__close30_minus_full', 'mlofi__last', 'mlofi_flip_rate', 'nofi_level_5__abs_mean', 'nofi_level_3__abs_mean', 'nofi_level_2__abs_mean', 'nofi_level_1__abs_mean', 'near_minus_far_ofi__sum', 'nofi_level_5__sum', 'far_ofi__sum', 'nofi_level_4__sum', 'nofi_level_3__sum', 'mlofi__sum', 'nofi_level_1__sum', 'nofi_level_2__sum', 'buy_shock_continuation_5__mean', 'shock_reversal_5__mean', 'close30_shock_rate', 'shock_rate', 'resiliency_composite', 'unabsorbed_flow', 'buy_shock_rate', 'absorbed_flow', 'shock_non_recovery_5__rate', 'shock_spread_recovery_5__mean', 'shock_depth_recovery_5__mean', 'sell_shock_continuation_5__mean', 'impact_r2_proxy', 'impact_lambda', 'lambda_x_mlofi_sum', 'top_depth_concentration__std', 'depth_centroid_asymmetry__std', 'ask_gap_bps__mean', 'bid_gap_bps__std', 'price_gap_asymmetry__std', 'bid_gap_bps__mean', 'top_depth_concentration__mean', 'ask_gap_bps__std']
    ALL_FEATURES = (
        [f"zoo__{column}" for column in E4_FEATURES]
        + [f"p1__{column}" for column in PHASE1_FEATURES]
        + [f"p3__{column}" for column in PHASE3_FEATURES]
    )
    if len(E4_FEATURES) != 66 or len(PHASE1_FEATURES) != 25:
        raise RuntimeError("Frozen feature contract is invalid.")

    def _as_timestamp(value):
        timestamp = pd.Timestamp(value)
        if timestamp.tzinfo is not None:
            timestamp = timestamp.tz_localize(None)
        return timestamp

    def _end_of_day(value):
        return _as_timestamp(value).normalize() + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)

    def _timestamp_string(value):
        return _as_timestamp(value).strftime("%Y-%m-%d %H:%M:%S")

    def _query_pool(query_start, query_end):
        frame = dai.query(
            f"SELECT date, instrument FROM {INSTRUMENT_TABLE}",
            filters={"date": [_timestamp_string(query_start), _timestamp_string(_end_of_day(query_end))]},
            compression=True,
        ).df()
        if frame.empty:
            raise RuntimeError("Historical CSI-1000 pool query returned no rows.")
        frame["date"] = pd.to_datetime(frame["date"]).dt.normalize()
        frame["instrument"] = frame["instrument"].astype(str)
        return frame.drop_duplicates(["date", "instrument"]).sort_values(["date", "instrument"]).reset_index(drop=True)

    def _query_daily_close(bar_table, query_start, query_end):
        sql = f"""
        SELECT
            date_trunc('day', date)::DATE AS trading_day,
            instrument,
            ARG_MAX(close, date) AS close
        FROM {bar_table}
        WHERE close > 0
        GROUP BY trading_day, instrument
        ORDER BY trading_day, instrument
        """
        frame = dai.query(
            sql,
            filters={"date": [_timestamp_string(query_start), _timestamp_string(_end_of_day(query_end))]},
            compression=True,
        ).df().rename(columns={"trading_day": "date"})
        if frame.empty:
            raise RuntimeError("Daily-close query returned no rows.")
        frame["date"] = pd.to_datetime(frame["date"]).dt.normalize()
        frame["instrument"] = frame["instrument"].astype(str)
        frame["close"] = pd.to_numeric(frame["close"], errors="coerce")
        return frame.dropna(subset=["date", "instrument", "close"]).drop_duplicates(["date", "instrument"], keep="last").sort_values(["instrument", "date"]).reset_index(drop=True)

    def _attach_raw_forward_return(close_frame):
        output = close_frame.copy()
        calendar = pd.Index(sorted(output["date"].dropna().unique()))
        next_date_map = {calendar[index]: calendar[index + 1] for index in range(len(calendar) - 1)}
        grouped = output.groupby("instrument", sort=False)
        output["target_date"] = grouped["date"].shift(-1)
        output["next_close"] = grouped["close"].shift(-1)
        output["expected_target_date"] = output["date"].map(next_date_map)
        valid = output["target_date"].eq(output["expected_target_date"]) & output["close"].gt(0) & output["next_close"].gt(0)
        output["target_return"] = np.where(valid, output["next_close"] / output["close"] - 1.0, np.nan)
        return output[["date", "instrument", "target_return"]]

    def _cs_z(frame, columns):
        output = frame[["date", "instrument", *columns]].copy()
        for column in columns:
            output[column] = pd.to_numeric(output[column], errors="coerce")
        grouped = output.groupby("date", sort=False)
        lower = grouped[columns].transform(lambda values: values.quantile(0.01))
        upper = grouped[columns].transform(lambda values: values.quantile(0.99))
        clipped = output[columns].clip(lower=lower, upper=upper)
        mean = clipped.groupby(output["date"], sort=False).transform("mean")
        std = clipped.groupby(output["date"], sort=False).transform("std").where(lambda values: values > 1e-12)
        output.loc[:, columns] = ((clipped - mean) / std).replace([np.inf, -np.inf], np.nan).astype(np.float32)
        return output

    def _sample_per_date(frame, maximum):
        pieces = []
        for _, block in frame.sort_values(["date", "instrument"]).groupby("date", sort=False):
            if len(block) <= maximum:
                pieces.append(block)
            else:
                positions = np.linspace(0, len(block) - 1, maximum, dtype=np.int64)
                pieces.append(block.iloc[positions])
        return pd.concat(pieces, ignore_index=True)

    def _build_e4_features(block_datasources, block_start, block_end):
        datasources = block_datasources
        start_date = block_start
        end_date = block_end
        """
        BigAlpha E4 Literature + WQ + GTJA XGBoost submission.

        Public training data are capped at 2019-01-01 through 2024-12-31. The E4 feature architecture is frozen exactly from Fold 3 (test year 2024) of the completed tournament.
        Both training and prediction read bar1m/financial tables exclusively
        through the platform-injected datasources mapping.
        """
        import gc
        import math
        import random
        import numpy as np
        import pandas as pd
        import dai
        import torch
        import torch.nn as nn

        if not isinstance(datasources, dict):
            raise TypeError("datasources must be a dictionary.")
        missing_sources = [
            key for key in ("bar1m", "financial")
            if key not in datasources
        ]
        if missing_sources:
            raise KeyError(
                f"datasources missing required keys: {missing_sources}"
            )

        DEFAULT_FACTORLIB_TABLE = "bigalpha_2026_factorlib"


        FINANCIAL_FIELDS = [
            "total_assets",
            "total_liabilities",
            "total_equity_to_parent_shareholders",
            "gross_profit",
            "operating_revenue",
            "operating_profit",
            "net_profit",
            "net_profit_to_parent_shareholders",
            "net_profit_to_parent_deducted",
            "net_cffoa",
            "inventories",
            "accounts_receivable",
            "fixed_assets",
            "moneytary_assets",
            "tradable_fin_assets",
            "interest_bearing_debt",
            "net_debt",
            "research_and_development_expense",
            "latest_shares",
            "longterm_borrowings",
            "bonds_payable",
            "total_current_assets",
            "total_current_liabilities",
            "capital_contributions_received",
        ]


        FACTORLIB_FIELDS = [
            "date",
            "instrument",
            "close",
            "amount",
            "turn",
            "daily_return",
            "reversal_5",
            "volatility_5",
            "total_market_cap",
            "float_market_cap",
            "sma_20",
            "ema_20",
            "macd_hist_12_26_9",
            "rsi_12",
            "bias_20",
            "atr_14",
            "gross_profit_rate_ttm",
            "current_ratio_lf",
            "netflow_amount_main",
            "netflow_amount_rate_main",
            "net_active_buy_amount_main",
            "list_days",
        ]


        TECHNICAL_WARMUP_CALENDAR_DAYS = 120


        def _as_timestamp(value):
            timestamp = pd.Timestamp(value)
            if timestamp.tzinfo is not None:
                timestamp = timestamp.tz_localize(None)
            return timestamp


        def _date_string(value):
            return _as_timestamp(value).strftime("%Y-%m-%d %H:%M:%S")


        def _safe_divide(numerator, denominator, min_abs_denominator=1e-12):
            numerator = pd.to_numeric(numerator, errors="coerce")
            denominator = pd.to_numeric(denominator, errors="coerce")
            valid = denominator.abs() > min_abs_denominator
            result = pd.Series(np.nan, index=numerator.index, dtype="float64")
            result.loc[valid] = numerator.loc[valid] / denominator.loc[valid]
            return result.replace([np.inf, -np.inf], np.nan)


        def _query_factorlib(table, query_start, query_end):
            fields = ",\n        ".join(FACTORLIB_FIELDS)
            sql = f"""
            SELECT
                {fields}
            FROM {table}
            """
            frame = dai.query(
                sql,
                filters={"date": [_date_string(query_start), _date_string(query_end)]},
                compression=True,
            ).df()
            if frame.empty:
                raise RuntimeError("factorlib query returned no rows.")
            frame["date"] = pd.to_datetime(frame["date"]).dt.normalize()
            frame["instrument"] = frame["instrument"].astype(str)
            return frame


        def _query_instruments(query_start, query_end):
            sql = """
            SELECT
                date,
                instrument
            FROM bigalpha_2026_instruments
            """
            frame = dai.query(
                sql,
                filters={"date": [_date_string(query_start), _date_string(query_end)]},
                compression=True,
            ).df()
            if frame.empty:
                raise RuntimeError("instrument-universe query returned no rows.")
            frame["date"] = pd.to_datetime(frame["date"]).dt.normalize()
            frame["instrument"] = frame["instrument"].astype(str)
            return frame.drop_duplicates(["date", "instrument"])


        def _query_financial(financial_table, financial_query_start, query_end):
            fields = ",\n        ".join(FINANCIAL_FIELDS)
            sql = f"""
            SELECT
                date,
                instrument,
                report_date,
                shift,
                category,
                {fields}
            FROM {financial_table}
            WHERE shift <= 11
            """
            frame = dai.query(
                sql,
                filters={"date": [_date_string(financial_query_start), _date_string(query_end)]},
                compression=True,
            ).df()
            if frame.empty:
                raise RuntimeError("financial query returned no rows.")
            frame["date"] = pd.to_datetime(frame["date"]).dt.normalize()
            frame["report_date"] = pd.to_datetime(frame["report_date"]).dt.normalize()
            frame["instrument"] = frame["instrument"].astype(str)
            frame["category"] = frame["category"].astype(str).str.lower()
            frame["shift"] = pd.to_numeric(frame["shift"], errors="coerce")
            return frame


        def _flatten_financial_rows(financial):
            keys = ["date", "instrument", "report_date", "shift"]
            financial = (
                financial.sort_values(keys + ["category"])
                .drop_duplicates(keys + ["category"], keep="last")
                .copy()
            )

            wide = (
                financial
                .set_index(keys + ["category"])[FINANCIAL_FIELDS]
                .unstack("category")
                .reset_index()
            )

            flattened = []
            for column in wide.columns:
                if isinstance(column, tuple):
                    field, category = column
                    flattened.append(f"{field}__{category}" if category else str(field))
                else:
                    flattened.append(str(column))
            wide.columns = flattened

            for field in FINANCIAL_FIELDS:
                for category in ("lf", "mrq", "ttm"):
                    name = f"{field}__{category}"
                    if name not in wide.columns:
                        wide[name] = np.nan
            return wide


        def _build_snapshot_features(financial):
            wide = _flatten_financial_rows(financial)
            snapshot = wide[["date", "instrument"]].drop_duplicates().copy()

            requested = {
                "gross_ttm_0": ("gross_profit", "ttm", 0),
                "revenue_ttm_0": ("operating_revenue", "ttm", 0),
                "operating_profit_ttm_0": ("operating_profit", "ttm", 0),
                "net_profit_ttm_0": ("net_profit", "ttm", 0),
                "parent_profit_ttm_0": ("net_profit_to_parent_shareholders", "ttm", 0),
                "parent_deducted_ttm_0": ("net_profit_to_parent_deducted", "ttm", 0),
                "cfo_ttm_0": ("net_cffoa", "ttm", 0),
                "rd_ttm_0": ("research_and_development_expense", "ttm", 0),
                "parent_profit_mrq_0": ("net_profit_to_parent_shareholders", "mrq", 0),
                "parent_profit_mrq_4": ("net_profit_to_parent_shareholders", "mrq", 4),
                "net_profit_mrq_0": ("net_profit", "mrq", 0),
                "net_profit_mrq_4": ("net_profit", "mrq", 4),
                "assets_lf_0": ("total_assets", "lf", 0),
                "assets_lf_1": ("total_assets", "lf", 1),
                "assets_lf_4": ("total_assets", "lf", 4),
                "assets_lf_5": ("total_assets", "lf", 5),
                "equity_parent_lf_1": ("total_equity_to_parent_shareholders", "lf", 1),
                "equity_parent_lf_5": ("total_equity_to_parent_shareholders", "lf", 5),
                "liabilities_lf_0": ("total_liabilities", "lf", 0),
                "cash_lf_0": ("moneytary_assets", "lf", 0),
                "trading_assets_lf_0": ("tradable_fin_assets", "lf", 0),
                "net_debt_lf_0": ("net_debt", "lf", 0),
            }

            for output_name, (field, category, shift_value) in requested.items():
                source_column = f"{field}__{category}"
                piece = (
                    wide.loc[
                        wide["shift"].eq(shift_value),
                        ["date", "instrument", source_column],
                    ]
                    .rename(columns={source_column: output_name})
                    .drop_duplicates(["date", "instrument"], keep="last")
                )
                snapshot = snapshot.merge(piece, on=["date", "instrument"], how="left")

            annual = wide.loc[
                wide["report_date"].dt.month.eq(12)
                & wide["report_date"].dt.day.eq(31)
            ].copy()
            annual = annual.sort_values(
                ["date", "instrument", "report_date"],
                ascending=[True, True, False],
            )
            annual["annual_rank"] = annual.groupby(
                ["date", "instrument"], sort=False
            ).cumcount()
            annual = annual.loc[annual["annual_rank"].le(2)]

            annual_fields = [
                "total_assets", "total_liabilities", "gross_profit",
                "operating_revenue", "net_profit", "net_cffoa", "inventories",
                "accounts_receivable", "fixed_assets", "moneytary_assets",
                "tradable_fin_assets", "interest_bearing_debt", "latest_shares",
                "longterm_borrowings", "bonds_payable", "total_current_assets",
                "total_current_liabilities", "capital_contributions_received",
            ]

            for annual_rank in (0, 1, 2):
                source = annual.loc[
                    annual["annual_rank"].eq(annual_rank),
                    ["date", "instrument", "report_date"]
                    + [f"{field}__lf" for field in annual_fields],
                ].copy()
                rename = {"report_date": f"annual_report_date_{annual_rank}"}
                rename.update({
                    f"{field}__lf": f"{field}_ann{annual_rank}"
                    for field in annual_fields
                })
                source = source.rename(columns=rename)
                snapshot = snapshot.merge(
                    source,
                    on=["date", "instrument"],
                    how="left",
                )

            snapshot["GPA_TTM"] = _safe_divide(
                snapshot["gross_ttm_0"], snapshot["assets_lf_0"]
            )
            snapshot["GLA_TTM"] = _safe_divide(
                snapshot["gross_ttm_0"], snapshot["assets_lf_4"]
            )

            roe_current = _safe_divide(
                snapshot["parent_profit_mrq_0"], snapshot["equity_parent_lf_1"]
            )
            roe_last_year = _safe_divide(
                snapshot["parent_profit_mrq_4"], snapshot["equity_parent_lf_5"]
            )
            roa_current = _safe_divide(
                snapshot["net_profit_mrq_0"], snapshot["assets_lf_1"]
            )
            roa_last_year = _safe_divide(
                snapshot["net_profit_mrq_4"], snapshot["assets_lf_5"]
            )

            snapshot["ROE_Q"] = roe_current
            snapshot["ROA_Q"] = roa_current
            snapshot["D_ROE_4Q"] = roe_current - roe_last_year
            snapshot["D_ROA_4Q"] = roa_current - roa_last_year
            snapshot["CFOA_LAG"] = _safe_divide(
                snapshot["cfo_ttm_0"], snapshot["assets_lf_4"]
            )
            snapshot["PROFIT_MARGIN"] = _safe_divide(
                snapshot["net_profit_ttm_0"], snapshot["revenue_ttm_0"]
            )
            snapshot["ASSET_TURNOVER"] = _safe_divide(
                snapshot["revenue_ttm_0"], snapshot["assets_lf_4"]
            )
            snapshot["OPA_LAG"] = _safe_divide(
                snapshot["operating_profit_ttm_0"], snapshot["assets_lf_4"]
            )

            accrual_amount = snapshot["net_profit_ttm_0"] - snapshot["cfo_ttm_0"]
            snapshot["OA_CF"] = _safe_divide(accrual_amount, snapshot["assets_lf_4"])
            snapshot["POA"] = _safe_divide(
                accrual_amount, snapshot["net_profit_ttm_0"].abs()
            )
            snapshot["BOOK_LEVERAGE"] = _safe_divide(
                snapshot["liabilities_lf_0"], snapshot["assets_lf_0"]
            )
            snapshot["CASH_TO_ASSETS"] = _safe_divide(
                snapshot["cash_lf_0"].fillna(0)
                + snapshot["trading_assets_lf_0"].fillna(0),
                snapshot["assets_lf_0"],
            )

            assets0 = snapshot["total_assets_ann0"]
            assets1 = snapshot["total_assets_ann1"]
            assets2 = snapshot["total_assets_ann2"]

            snapshot["ASSET_GROWTH_YOY"] = _safe_divide(assets0, assets1) - 1.0
            snapshot["INVENTORY_GROWTH_YOY"] = (
                _safe_divide(snapshot["inventories_ann0"], snapshot["inventories_ann1"])
                - 1.0
            )
            snapshot["INVENTORY_CHANGE_A"] = _safe_divide(
                snapshot["inventories_ann0"] - snapshot["inventories_ann1"],
                (assets0 + assets1) / 2.0,
            )
            snapshot["RECEIVABLES_GROWTH"] = (
                _safe_divide(
                    snapshot["accounts_receivable_ann0"],
                    snapshot["accounts_receivable_ann1"],
                )
                - 1.0
            )
            snapshot["DPIA"] = _safe_divide(
                (
                    snapshot["fixed_assets_ann0"].fillna(0)
                    + snapshot["inventories_ann0"].fillna(0)
                )
                - (
                    snapshot["fixed_assets_ann1"].fillna(0)
                    + snapshot["inventories_ann1"].fillna(0)
                ),
                assets1,
            )

            operating_assets = (
                assets0
                - snapshot["moneytary_assets_ann0"].fillna(0)
                - snapshot["tradable_fin_assets_ann0"].fillna(0)
            )
            operating_liabilities = (
                snapshot["total_liabilities_ann0"]
                - snapshot["interest_bearing_debt_ann0"].fillna(0)
            )
            snapshot["NOA"] = _safe_divide(
                operating_assets - operating_liabilities,
                assets1,
            )
            snapshot["NET_STOCK_ISSUES"] = np.log(
                _safe_divide(
                    snapshot["latest_shares_ann0"],
                    snapshot["latest_shares_ann1"],
                )
            ).replace([np.inf, -np.inf], np.nan)

            avg_assets_01 = (assets0 + assets1) / 2.0
            avg_assets_12 = (assets1 + assets2) / 2.0
            annual_roa0 = _safe_divide(snapshot["net_profit_ann0"], assets1)
            annual_roa1 = _safe_divide(snapshot["net_profit_ann1"], assets2)
            annual_cfo0 = _safe_divide(snapshot["net_cffoa_ann0"], assets1)

            debt0 = (
                snapshot["longterm_borrowings_ann0"].fillna(0)
                + snapshot["bonds_payable_ann0"].fillna(0)
            )
            debt1 = (
                snapshot["longterm_borrowings_ann1"].fillna(0)
                + snapshot["bonds_payable_ann1"].fillna(0)
            )
            leverage0 = _safe_divide(debt0, avg_assets_01)
            leverage1 = _safe_divide(debt1, avg_assets_12)
            current_ratio0 = _safe_divide(
                snapshot["total_current_assets_ann0"],
                snapshot["total_current_liabilities_ann0"],
            )
            current_ratio1 = _safe_divide(
                snapshot["total_current_assets_ann1"],
                snapshot["total_current_liabilities_ann1"],
            )
            gross_margin0 = _safe_divide(
                snapshot["gross_profit_ann0"],
                snapshot["operating_revenue_ann0"],
            )
            gross_margin1 = _safe_divide(
                snapshot["gross_profit_ann1"],
                snapshot["operating_revenue_ann1"],
            )
            turnover0 = _safe_divide(
                snapshot["operating_revenue_ann0"], avg_assets_01
            )
            turnover1 = _safe_divide(
                snapshot["operating_revenue_ann1"], avg_assets_12
            )

            signals = pd.DataFrame(index=snapshot.index)
            signals["F_ROA"] = annual_roa0.gt(0).where(annual_roa0.notna())
            signals["F_CFO"] = annual_cfo0.gt(0).where(annual_cfo0.notna())
            signals["F_DROA"] = annual_roa0.gt(annual_roa1).where(
                annual_roa0.notna() & annual_roa1.notna()
            )
            signals["F_ACCRUAL"] = annual_cfo0.gt(annual_roa0).where(
                annual_cfo0.notna() & annual_roa0.notna()
            )
            signals["F_DLEVER"] = leverage0.lt(leverage1).where(
                leverage0.notna() & leverage1.notna()
            )
            signals["F_DLIQUID"] = current_ratio0.gt(current_ratio1).where(
                current_ratio0.notna() & current_ratio1.notna()
            )
            share_ratio = _safe_divide(
                snapshot["latest_shares_ann0"], snapshot["latest_shares_ann1"]
            )
            signals["EQOFFER"] = share_ratio.le(1.000001).where(share_ratio.notna())
            signals["F_DMARGIN"] = gross_margin0.gt(gross_margin1).where(
                gross_margin0.notna() & gross_margin1.notna()
            )
            signals["F_DTURN"] = turnover0.gt(turnover1).where(
                turnover0.notna() & turnover1.notna()
            )

            signal_count = signals.notna().sum(axis=1)
            snapshot["F_SCORE_COMPONENTS"] = signal_count
            snapshot["F_SCORE"] = (
                signals.astype(float).sum(axis=1, min_count=1)
                / signal_count.replace(0, np.nan)
                * 9.0
            ).where(signal_count.ge(7))

            output_columns = [
                "date", "instrument", "annual_report_date_0", "annual_report_date_1",
                "GPA_TTM", "GLA_TTM", "ROE_Q", "ROA_Q", "D_ROE_4Q", "D_ROA_4Q",
                "CFOA_LAG", "PROFIT_MARGIN", "ASSET_TURNOVER", "OPA_LAG",
                "OA_CF", "POA", "ASSET_GROWTH_YOY", "INVENTORY_GROWTH_YOY",
                "INVENTORY_CHANGE_A", "RECEIVABLES_GROWTH", "DPIA", "NOA",
                "NET_STOCK_ISSUES", "BOOK_LEVERAGE", "CASH_TO_ASSETS",
                "parent_deducted_ttm_0", "cfo_ttm_0", "net_debt_lf_0",
                "revenue_ttm_0", "rd_ttm_0", "F_SCORE", "F_SCORE_COMPONENTS",
            ]
            return snapshot[output_columns].drop_duplicates(["date", "instrument"])


        def _prepare_factorlib(factorlib):
            factorlib = (
                factorlib.sort_values(["instrument", "date"])
                .drop_duplicates(["date", "instrument"], keep="last")
                .copy()
            )
            factorlib["market_cap_lag1"] = (
                factorlib.groupby("instrument", sort=False)["total_market_cap"].shift(1)
            )
            factorlib["sma_gap_20"] = (
                _safe_divide(factorlib["close"], factorlib["sma_20"]) - 1.0
            )
            factorlib["ema_gap_20"] = (
                _safe_divide(factorlib["close"], factorlib["ema_20"]) - 1.0
            )
            factorlib["macd_hist_scaled"] = _safe_divide(
                factorlib["macd_hist_12_26_9"], factorlib["close"]
            )
            factorlib["atr_pct"] = _safe_divide(
                factorlib["atr_14"], factorlib["close"]
            )
            factorlib["main_flow_scaled"] = _safe_divide(
                factorlib["netflow_amount_main"], factorlib["amount"]
            )
            factorlib["active_buy_scaled"] = _safe_divide(
                factorlib["net_active_buy_amount_main"], factorlib["amount"]
            )
            factorlib["log_list_days"] = np.log1p(
                pd.to_numeric(factorlib["list_days"], errors="coerce").clip(lower=0)
            )
            return factorlib


        def _asof_join_daily_with_financial(daily, snapshot_features):
            left = daily.sort_values(["date", "instrument"]).copy()
            right = snapshot_features.rename(columns={"date": "financial_date"}).copy()
            right = right.sort_values(["financial_date", "instrument"])
            merged = pd.merge_asof(
                left,
                right,
                left_on="date",
                right_on="financial_date",
                by="instrument",
                direction="backward",
                allow_exact_matches=False,
            )
            return merged.sort_values(["date", "instrument"]).reset_index(drop=True)


        def build_feature_panel(
            start_date,
            end_date,
            factorlib_table=None,
            financial_table=None,
        ):
            target_start = _as_timestamp(start_date).normalize()
            target_end = _as_timestamp(end_date).normalize()
            if target_end < target_start:
                raise ValueError("end_date must be on or after start_date.")

            table = factorlib_table or DEFAULT_FACTORLIB_TABLE
            if not financial_table:
                raise ValueError("financial_table is required.")

            factorlib_query_start = target_start - pd.Timedelta(days=40)
            financial_query_start = target_start - pd.Timedelta(days=550)

            factorlib = _query_factorlib(
                table,
                factorlib_query_start,
                target_end,
            )
            universe = _query_instruments(
                factorlib_query_start,
                target_end,
            )

            # The official instrument universe is the left-hand spine.
            # Missing factorlib rows must not remove constituents or dates.
            daily = universe.merge(
                factorlib,
                on=["date", "instrument"],
                how="left",
                validate="one_to_one",
            )
            daily = _prepare_factorlib(daily)
            financial = _query_financial(
                financial_table,
                financial_query_start,
                target_end,
            )
            snapshots = _build_snapshot_features(financial)
            panel = _asof_join_daily_with_financial(daily, snapshots)

            panel["EP_CHINA"] = _safe_divide(
                panel["parent_deducted_ttm_0"], panel["market_cap_lag1"]
            )
            panel["OCFP"] = _safe_divide(
                panel["cfo_ttm_0"], panel["market_cap_lag1"]
            )
            panel["NDP"] = _safe_divide(
                panel["net_debt_lf_0"], panel["market_cap_lag1"]
            )
            panel["SP"] = _safe_divide(
                panel["revenue_ttm_0"], panel["market_cap_lag1"]
            )
            panel["RDM"] = _safe_divide(
                panel["rd_ttm_0"], panel["market_cap_lag1"]
            )

            panel = panel.loc[
                panel["date"].between(target_start, target_end)
            ].copy()

            if panel.duplicated(["date", "instrument"]).any():
                duplicates = int(panel.duplicated(["date", "instrument"]).sum())
                raise RuntimeError(
                    f"Feature panel contains {duplicates} duplicate date-instrument rows."
                )
            return panel.sort_values(["date", "instrument"]).reset_index(drop=True)


        def _daily_oriented_rank(frame, column, direction):
            values = pd.to_numeric(frame[column], errors="coerce") * float(direction)
            return values.groupby(frame["date"]).rank(
                method="average",
                pct=True,
                na_option="keep",
            )


        def build_literature_composite(panel, mode="category_equal"):
            frame = panel.copy()
            groups = {
                "profitability_quality": {
                    "GPA_TTM": 1, "GLA_TTM": 1, "ROE_Q": 1, "ROA_Q": 1,
                    "D_ROE_4Q": 1, "D_ROA_4Q": 1, "CFOA_LAG": 1,
                    "PROFIT_MARGIN": 1, "ASSET_TURNOVER": 1, "OPA_LAG": 1,
                    "F_SCORE": 1, "CASH_TO_ASSETS": 1,
                    "gross_profit_rate_ttm": 1, "current_ratio_lf": 1,
                },
                "investment_and_accruals": {
                    "OA_CF": -1, "POA": -1, "ASSET_GROWTH_YOY": -1,
                    "INVENTORY_GROWTH_YOY": -1, "INVENTORY_CHANGE_A": -1,
                    "RECEIVABLES_GROWTH": -1, "DPIA": -1, "NOA": -1,
                    "NET_STOCK_ISSUES": -1,
                },
                "value": {
                    "EP_CHINA": 1, "OCFP": 1, "NDP": 1, "SP": 1,
                },
                "trading_and_friction": {
                    "reversal_5": 1, "volatility_5": -1, "turn": -1,
                    "atr_pct": -1, "netflow_amount_rate_main": 1,
                },
            }

            all_rank_columns = []
            category_columns = []
            for group_name, feature_directions in groups.items():
                group_rank_columns = []
                for feature, direction in feature_directions.items():
                    if feature not in frame.columns:
                        continue
                    rank_column = f"rank__{feature}"
                    frame[rank_column] = _daily_oriented_rank(
                        frame, feature, direction
                    )
                    group_rank_columns.append(rank_column)
                    all_rank_columns.append(rank_column)

                category_column = f"category__{group_name}"
                available_count = frame[group_rank_columns].notna().sum(axis=1)
                minimum_features = max(2, int(np.ceil(len(group_rank_columns) * 0.30)))
                frame[category_column] = frame[group_rank_columns].mean(
                    axis=1, skipna=True
                ).where(available_count.ge(minimum_features))
                category_columns.append(category_column)

            if mode == "category_equal":
                raw_score = frame[category_columns].mean(axis=1, skipna=True).where(
                    frame[category_columns].notna().sum(axis=1).ge(1)
                )
            else:
                raw_score = frame[all_rank_columns].mean(axis=1, skipna=True)

            frame["factor"] = raw_score.groupby(frame["date"]).rank(
                method="average", pct=True, na_option="keep"
            ) - 0.5
            frame["factor"] = frame.groupby("date")["factor"].transform(
                lambda series: series.fillna(series.median())
            ).fillna(0.0)
            return frame


        def _query_daily_ohlcv(bar1m_table, query_start, query_end):
            sql = f"""
            SELECT
                date::DATE::DATETIME AS date,
                instrument,
                FIRST(open ORDER BY date) AS bar_open,
                MAX(high) AS bar_high,
                MIN(low) AS bar_low,
                LAST(close ORDER BY date) AS bar_close,
                FIRST(pre_close ORDER BY date) AS bar_pre_close,
                MAX(volume) AS bar_volume,
                MAX(amount) AS bar_amount
            FROM {bar1m_table}
            GROUP BY date::DATE, instrument
            ORDER BY date, instrument
            """
            # bar1m timestamps are intraday. Passing an end date at 00:00:00
            # can exclude that entire final trading day. Query through the next
            # midnight, then rely on the universe/output filter to keep only
            # dates <= query_end. This is required for prefix invariance.
            query_end_exclusive = (
                _as_timestamp(query_end).normalize() + pd.Timedelta(days=1)
            )
            frame = dai.query(
                sql,
                filters={
                    "date": [
                        _date_string(query_start),
                        _date_string(query_end_exclusive),
                    ]
                },
                compression=True,
            ).df()
            if frame.empty:
                raise RuntimeError("Daily OHLCV aggregation returned no rows.")
            frame["date"] = pd.to_datetime(frame["date"]).dt.normalize()
            frame["instrument"] = frame["instrument"].astype(str)
            for column in [
                "bar_open", "bar_high", "bar_low", "bar_close", "bar_pre_close",
                "bar_volume", "bar_amount",
            ]:
                frame[column] = pd.to_numeric(frame[column], errors="coerce")
            frame["bar_vwap"] = _safe_divide(frame["bar_amount"], frame["bar_volume"])
            frame["bar_return"] = _safe_divide(
                frame["bar_close"] - frame["bar_pre_close"], frame["bar_pre_close"]
            )
            return frame.drop_duplicates(["date", "instrument"], keep="last")


        def _groups(frame):
            return frame.groupby("instrument", sort=False)


        def _ts_delay(series, groups, window):
            return groups[series.name].shift(window) if series.name in groups.obj else (
                series.groupby(groups.obj["instrument"], sort=False).shift(window)
            )


        def _ts_delta(series, groups, window):
            return series - series.groupby(
                groups.obj["instrument"], sort=False
            ).shift(window)


        def _ts_sum(series, groups, window):
            return series.groupby(groups.obj["instrument"], sort=False).transform(
                lambda values: values.rolling(window, min_periods=window).sum()
            )


        def _ts_mean(series, groups, window):
            return series.groupby(groups.obj["instrument"], sort=False).transform(
                lambda values: values.rolling(window, min_periods=window).mean()
            )


        def _ts_std(series, groups, window):
            return series.groupby(groups.obj["instrument"], sort=False).transform(
                lambda values: values.rolling(window, min_periods=window).std(ddof=0)
            )


        def _ts_min(series, groups, window):
            return series.groupby(groups.obj["instrument"], sort=False).transform(
                lambda values: values.rolling(window, min_periods=window).min()
            )


        def _ts_max(series, groups, window):
            return series.groupby(groups.obj["instrument"], sort=False).transform(
                lambda values: values.rolling(window, min_periods=window).max()
            )


        def _ts_corr(left, right, groups, window):
            instrument = groups.obj["instrument"]
            return pd.concat(
                [left.rename("_left"), right.rename("_right"), instrument],
                axis=1,
            ).groupby("instrument", sort=False, group_keys=False).apply(
                lambda block: block["_left"].rolling(
                    window, min_periods=window
                ).corr(block["_right"])
            ).reset_index(level=0, drop=True).reindex(left.index)


        def _ts_cov(left, right, groups, window):
            instrument = groups.obj["instrument"]
            return pd.concat(
                [left.rename("_left"), right.rename("_right"), instrument],
                axis=1,
            ).groupby("instrument", sort=False, group_keys=False).apply(
                lambda block: block["_left"].rolling(
                    window, min_periods=window
                ).cov(block["_right"])
            ).reset_index(level=0, drop=True).reindex(left.index)


        def _ts_argmax(series, groups, window):
            return series.groupby(groups.obj["instrument"], sort=False).transform(
                lambda values: values.rolling(window, min_periods=window).apply(
                    lambda array: float(np.argmax(array) + 1), raw=True
                )
            )


        def _cs_rank(series, dates):
            return pd.to_numeric(series, errors="coerce").groupby(dates).rank(
                method="average", pct=True, na_option="keep"
            )


        def _signed_power(series, exponent):
            values = pd.to_numeric(series, errors="coerce")
            return np.sign(values) * np.power(np.abs(values), exponent)


        def _safe_formula_div(numerator, denominator, epsilon=1e-12):
            numerator = pd.to_numeric(numerator, errors="coerce")
            denominator = pd.to_numeric(denominator, errors="coerce")
            result = numerator / denominator.where(denominator.abs().gt(epsilon))
            return result.replace([np.inf, -np.inf], np.nan)


        def compute_worldquant_batch(frame):
            f = frame
            g = _groups(f)
            d = f["date"]
            o, h, l, c = f["bar_open"], f["bar_high"], f["bar_low"], f["bar_close"]
            v, vw, ret = f["bar_volume"], f["bar_vwap"], f["bar_return"]
            adv20 = _ts_mean(v, g, 20)
            logv = np.log(v.clip(lower=1.0))
            candle = _safe_formula_div(c - o, o)

            base1 = pd.Series(np.where(ret < 0, _ts_std(ret, g, 20), c), index=f.index)
            f["wq_001"] = _cs_rank(_ts_argmax(_signed_power(base1, 2), g, 5), d)
            f["wq_002"] = -_ts_corr(
                _cs_rank(_ts_delta(logv, g, 2), d), _cs_rank(candle, d), g, 6
            )
            f["wq_003"] = -_ts_corr(_cs_rank(o, d), _cs_rank(v, d), g, 10)
            f["wq_005"] = _cs_rank(o - _ts_mean(vw, g, 10), d) * (
                -np.abs(_cs_rank(c - vw, d))
            )
            f["wq_006"] = -_ts_corr(o, v, g, 10)
            f["wq_012"] = np.sign(_ts_delta(v, g, 1)) * (-_ts_delta(c, g, 1))
            f["wq_013"] = -_cs_rank(
                _ts_cov(_cs_rank(c, d), _cs_rank(v, d), g, 5), d
            )
            f["wq_014"] = -_cs_rank(_ts_delta(ret, g, 3), d) * _ts_corr(o, v, g, 10)
            f["wq_015"] = -_ts_sum(
                _cs_rank(_ts_corr(_cs_rank(h, d), _cs_rank(v, d), g, 3), d), g, 3
            )
            f["wq_016"] = -_cs_rank(
                _ts_cov(_cs_rank(h, d), _cs_rank(v, d), g, 5), d
            )
            f["wq_018"] = -_cs_rank(
                _ts_std(np.abs(c - o), g, 5) + (c - o) + _ts_corr(c, o, g, 10), d
            )
            f["wq_020"] = (
                -_cs_rank(o - _ts_delay(h, g, 1), d)
                * _cs_rank(o - _ts_delay(c, g, 1), d)
                * _cs_rank(o - _ts_delay(l, g, 1), d)
            )
            f["wq_025"] = _cs_rank((-ret) * adv20 * vw * (h - c), d)
            f["wq_033"] = _cs_rank(-(1.0 - _safe_formula_div(o, c)), d)
            f["wq_034"] = _cs_rank(
                (1.0 - _cs_rank(
                    _safe_formula_div(_ts_std(ret, g, 2), _ts_std(ret, g, 5)), d
                ))
                + (1.0 - _cs_rank(_ts_delta(c, g, 1), d)), d
            )
            f["wq_040"] = -_cs_rank(_ts_std(h, g, 10), d) * _ts_corr(h, v, g, 10)
            f["wq_041"] = np.sqrt((h * l).clip(lower=0)) - vw
            f["wq_042"] = _safe_formula_div(
                _cs_rank(vw - c, d), _cs_rank(vw + c, d)
            )
            f["wq_044"] = -_ts_corr(h, _cs_rank(v, d), g, 5)
            candle53 = _safe_formula_div((c - l) - (h - c), c - l)
            f["wq_053"] = -_ts_delta(candle53, g, 9)
            f["wq_054"] = -_safe_formula_div(
                (l - c) * (o ** 5), (l - h) * (c ** 5)
            )
            stochastic = _safe_formula_div(
                c - _ts_min(l, g, 12),
                _ts_max(h, g, 12) - _ts_min(l, g, 12),
            )
            f["wq_055"] = -_ts_corr(
                _cs_rank(stochastic, d), _cs_rank(v, d), g, 6
            )
            return f




        def _days_since_extreme(series, groups, window, mode="max"):
            window = int(window)
            chooser = np.argmax if mode == "max" else np.argmin
            instrument = groups.obj["instrument"]
            return series.groupby(instrument, sort=False).transform(
                lambda values: values.rolling(window, min_periods=window).apply(
                    lambda x: float(window - 1 - chooser(x)), raw=True
                )
            )

        def compute_gtja_batch(frame):
            f = frame
            g = _groups(f)
            d = f["date"]
            o, h, l, c = f["bar_open"], f["bar_high"], f["bar_low"], f["bar_close"]
            v, vw, ret, amount = (
                f["bar_volume"], f["bar_vwap"], f["bar_return"], f["bar_amount"]
            )
            prev_c = _ts_delay(c, g, 1)
            delta_c = c - prev_c
            candle_imbalance = _safe_formula_div((c - l) - (h - c), h - l)

            f["gtja_002"] = -_ts_delta(candle_imbalance, g, 1)
            adjusted = pd.Series(
                np.where(
                    c.eq(prev_c),
                    0.0,
                    np.where(c.gt(prev_c), c - np.minimum(l, prev_c), c - np.maximum(h, prev_c)),
                ),
                index=f.index,
            )
            f["gtja_003"] = _ts_sum(adjusted, g, 6)
            mean8, std8, mean2 = _ts_mean(c, g, 8), _ts_std(c, g, 8), _ts_mean(c, g, 2)
            volume_ratio = _safe_formula_div(v, _ts_mean(v, g, 20))
            f["gtja_004"] = pd.Series(
                np.where(
                    mean8 + std8 < mean2,
                    -1.0,
                    np.where(mean2 < mean8 - std8, 1.0, np.where(volume_ratio >= 1.0, 1.0, -1.0)),
                ),
                index=f.index,
            )
            f["gtja_006"] = -_cs_rank(np.sign(_ts_delta(0.85 * o + 0.15 * h, g, 4)), d)
            f["gtja_008"] = -_cs_rank(_ts_delta(0.2 * ((h + l) / 2.0) + 0.8 * vw, g, 4), d)
            f["gtja_011"] = _ts_sum(candle_imbalance * v, g, 6)
            f["gtja_014"] = c - _ts_delay(c, g, 5)
            f["gtja_015"] = _safe_formula_div(o, _ts_delay(c, g, 1)) - 1.0
            f["gtja_020"] = 100.0 * _safe_formula_div(c - _ts_delay(c, g, 6), _ts_delay(c, g, 6))
            mean12 = _ts_mean(c, g, 12)
            f["gtja_031"] = 100.0 * _safe_formula_div(c - mean12, mean12)
            f["gtja_034"] = _safe_formula_div(mean12, c)
            interaction = _ts_sum(o, g, 5) * _ts_sum(ret, g, 5)
            f["gtja_037"] = -_cs_rank(interaction - _ts_delay(interaction, g, 10), d)
            f["gtja_038"] = pd.Series(
                np.where(_ts_mean(h, g, 20) < h, -_ts_delta(h, g, 2), 0.0), index=f.index
            )
            up_volume = pd.Series(np.where(c > prev_c, v, 0.0), index=f.index)
            down_volume = pd.Series(np.where(c <= prev_c, v, 0.0), index=f.index)
            f["gtja_040"] = 100.0 * _safe_formula_div(
                _ts_sum(up_volume, g, 26), _ts_sum(down_volume, g, 26)
            )
            signed_volume = pd.Series(np.sign(delta_c) * v, index=f.index)
            f["gtja_043"] = _ts_sum(signed_volume, g, 6)
            f["gtja_046"] = (
                _ts_mean(c, g, 3) + _ts_mean(c, g, 6) + _ts_mean(c, g, 12) + _ts_mean(c, g, 24)
            ) / (4.0 * c)
            f["gtja_053"] = 100.0 * _ts_mean((c > prev_c).astype(float), g, 12)
            f["gtja_070"] = _ts_std(amount, g, 6)
            f["gtja_080"] = 100.0 * _safe_formula_div(v - _ts_delay(v, g, 5), _ts_delay(v, g, 5))
            f["gtja_095"] = _ts_std(amount, g, 20)
            corr_hv5 = _ts_corr(h, v, g, 5)
            f["gtja_104"] = -_ts_delta(corr_hv5, g, 5) * _cs_rank(_ts_std(c, g, 20), d)
            f["gtja_110"] = 100.0 * _safe_formula_div(
                _ts_sum(np.maximum(0.0, h - prev_c), g, 20),
                _ts_sum(np.maximum(0.0, prev_c - l), g, 20),
            )
            positive_change = np.maximum(delta_c, 0.0)
            negative_change = np.maximum(-delta_c, 0.0)
            f["gtja_112"] = 100.0 * _safe_formula_div(
                _ts_sum(positive_change, g, 12) - _ts_sum(negative_change, g, 12),
                _ts_sum(positive_change, g, 12) + _ts_sum(negative_change, g, 12),
            )
            f["gtja_118"] = 100.0 * _safe_formula_div(
                _ts_sum(h - o, g, 20), _ts_sum(o - l, g, 20)
            )
            highday = _days_since_extreme(h, g, 20, mode="max")
            lowday = _days_since_extreme(l, g, 20, mode="min")
            f["gtja_133"] = ((20.0 - highday) / 20.0) * 100.0 - ((20.0 - lowday) / 20.0) * 100.0
            f["gtja_136"] = -_cs_rank(_ts_delta(ret, g, 3), d) * _ts_corr(o, v, g, 10)
            return f

        # ============================================================
        # Public training is capped at 2024-12-31. For an earlier local
        # smoke-test interval, the training end automatically moves backward.
        # ============================================================
        TRAIN_START = pd.Timestamp("2019-01-01")
        TRAIN_END_CAP = pd.Timestamp("2024-12-31")
        TRAIN_FACTORLIB_TABLE = "bigalpha_2026_factorlib"
        TRAIN_BAR1M_TABLE = datasources["bar1m"]
        TRAIN_FINANCIAL_TABLE = datasources["financial"]
        EXPOSURE_TABLE = "bigalpha_2026_exposure"
        TRAIN_EPOCHS = 40
        RANDOM_SEED = 17
        INCLUDE_GTJA = True

        CURRENT_FACTOR_GROUPS = {'profitability_quality': {'GPA_TTM': 1, 'GLA_TTM': 1, 'ROE_Q': 1, 'ROA_Q': 1, 'D_ROE_4Q': 1, 'D_ROA_4Q': 1, 'CFOA_LAG': 1, 'PROFIT_MARGIN': 1, 'ASSET_TURNOVER': 1, 'OPA_LAG': 1, 'F_SCORE': 1, 'CASH_TO_ASSETS': 1, 'gross_profit_rate_ttm': 1, 'current_ratio_lf': 1}, 'investment_and_accruals': {'OA_CF': -1, 'POA': -1, 'ASSET_GROWTH_YOY': -1, 'INVENTORY_GROWTH_YOY': -1, 'INVENTORY_CHANGE_A': -1, 'RECEIVABLES_GROWTH': -1, 'DPIA': -1, 'NOA': -1, 'NET_STOCK_ISSUES': -1}, 'value': {'EP_CHINA': 1, 'OCFP': 1, 'NDP': 1, 'SP': 1}, 'trading_and_friction': {'reversal_5': 1, 'volatility_5': -1, 'turn': -1, 'atr_pct': -1, 'netflow_amount_rate_main': 1}}
        PUBLIC_POOL_PROXY_COLUMNS = ['turn', 'reversal_5', 'volatility_5', 'total_market_cap', 'float_market_cap', 'rsi_12', 'bias_20', 'atr_pct', 'gross_profit_rate_ttm', 'current_ratio_lf', 'netflow_amount_rate_main', 'list_days']
        FALLBACK_RISK_COLUMNS = ['total_market_cap', 'float_market_cap', 'turn', 'reversal_5', 'volatility_5', 'bias_20', 'gross_profit_rate_ttm']
        WQ_META = {'wq_001': ('volatility_range', 20, 'rank(ts_argmax(signedpower(if ret<0 then std20(ret) else close,2),5))'), 'wq_002': ('price_volume_divergence', 8, '-corr(rank(delta(log(volume),2)),rank((close-open)/open),6)'), 'wq_003': ('price_volume_divergence', 10, '-corr(rank(open),rank(volume),10)'), 'wq_005': ('gap_intraday', 10, 'rank(open-mean(vwap,10))*-abs(rank(close-vwap))'), 'wq_006': ('price_volume_divergence', 10, '-corr(open,volume,10)'), 'wq_012': ('reversal_momentum', 2, 'sign(delta(volume,1))*-delta(close,1)'), 'wq_013': ('price_volume_divergence', 5, '-rank(cov(rank(close),rank(volume),5))'), 'wq_014': ('price_volume_divergence', 13, '-rank(delta(ret,3))*corr(open,volume,10)'), 'wq_015': ('price_volume_divergence', 6, '-sum(rank(corr(rank(high),rank(volume),3)),3)'), 'wq_016': ('price_volume_divergence', 5, '-rank(cov(rank(high),rank(volume),5))'), 'wq_018': ('gap_intraday', 10, '-rank(std(abs(close-open),5)+(close-open)+corr(close,open,10))'), 'wq_020': ('gap_intraday', 2, '-rank(open-delay(high,1))*rank(open-delay(close,1))*rank(open-delay(low,1))'), 'wq_025': ('liquidity_turnover', 20, 'rank(-ret*adv20*vwap*(high-close))'), 'wq_033': ('gap_intraday', 1, 'rank(-(1-open/close))'), 'wq_034': ('reversal_momentum', 5, 'rank((1-rank(std(ret,2)/std(ret,5)))+(1-rank(delta(close,1))))'), 'wq_040': ('volatility_range', 10, '-rank(std(high,10))*corr(high,volume,10)'), 'wq_041': ('gap_intraday', 1, 'sqrt(high*low)-vwap'), 'wq_042': ('gap_intraday', 1, 'rank(vwap-close)/rank(vwap+close)'), 'wq_044': ('price_volume_divergence', 5, '-corr(high,rank(volume),5)'), 'wq_053': ('gap_intraday', 10, '-delta(((close-low)-(high-close))/(close-low),9)'), 'wq_054': ('gap_intraday', 1, '-((low-close)*open^5)/((low-high)*close^5)'), 'wq_055': ('price_volume_divergence', 17, '-corr(rank(stochastic12),rank(volume),6)')}



        GTJA_META = {
            "gtja_002": ("gap_intraday", 2, "-delta(candle_imbalance,1)"),
            "gtja_003": ("reversal_momentum", 7, "sum(adjusted_close_change,6)"),
            "gtja_004": ("reversal_momentum", 20, "MA/STD/volume regime rule"),
            "gtja_006": ("gap_intraday", 5, "-rank(sign(delta(0.85*open+0.15*high,4)))"),
            "gtja_008": ("gap_intraday", 5, "-rank(delta(0.2*mid+0.8*vwap,4))"),
            "gtja_011": ("price_volume_divergence", 6, "sum(candle_imbalance*volume,6)"),
            "gtja_014": ("reversal_momentum", 6, "close-delay(close,5)"),
            "gtja_015": ("gap_intraday", 2, "open/delay(close,1)-1"),
            "gtja_020": ("reversal_momentum", 7, "6-day return*100"),
            "gtja_031": ("reversal_momentum", 12, "12-day close bias"),
            "gtja_034": ("reversal_momentum", 12, "mean(close,12)/close"),
            "gtja_037": ("reversal_momentum", 15, "-rank(sum(open,5)*sum(ret,5)-delay(...,10))"),
            "gtja_038": ("volatility_range", 20, "if mean(high,20)<high then -delta(high,2) else 0"),
            "gtja_040": ("liquidity_turnover", 26, "up-volume/down-volume ratio,26"),
            "gtja_043": ("liquidity_turnover", 7, "sum(sign(close change)*volume,6)"),
            "gtja_046": ("reversal_momentum", 24, "average of MA3/6/12/24 divided by close"),
            "gtja_053": ("reversal_momentum", 13, "count(up days,12)/12"),
            "gtja_070": ("liquidity_turnover", 6, "std(amount,6)"),
            "gtja_080": ("liquidity_turnover", 6, "5-day volume growth"),
            "gtja_095": ("liquidity_turnover", 20, "std(amount,20)"),
            "gtja_104": ("price_volume_divergence", 20, "-delta(corr(high,volume,5),5)*rank(std(close,20))"),
            "gtja_110": ("volatility_range", 20, "sum(max(high-prevclose,0),20)/sum(max(prevclose-low,0),20)"),
            "gtja_112": ("reversal_momentum", 13, "12-day positive-minus-negative price change ratio"),
            "gtja_118": ("gap_intraday", 20, "sum(high-open,20)/sum(open-low,20)"),
            "gtja_133": ("volatility_range", 20, "relative days since rolling high minus rolling low"),
            "gtja_136": ("price_volume_divergence", 13, "-rank(delta(ret,3))*corr(open,volume,10)"),
        }
        ALL_GTJA_FEATURE_COLUMNS = [
            'x__gtja_070',
            'x__gtja_095',
            'x__gtja_080',
            'x__gtja_043',
            'x__gtja_118',
            'x__gtja_046',
            'x__gtja_003',
            'x__gtja_031',
            'x__gtja_020',
            'x__gtja_040',
            'x__gtja_014',
            'x__gtja_038',
            'x__gtja_015',
            'x__gtja_002',
            'x__gtja_112',
            'x__gtja_008',
            'x__gtja_011',
            'x__gtja_110',
            'x__gtja_133',
            'x__gtja_037',
        ]

        # Fixed factor architecture selected before submission. Model coefficients
        # are NOT embedded: they are learned from scratch inside main on every run.
        E2_BASE_FEATURE_COLUMNS = ['x__current__turn', 'x__wq_016', 'x__wq_013', 'x__current__volatility_5', 'x__wq_054', 'x__current__atr_pct', 'x__wq_025', 'x__wq_055', 'x__wq_044', 'x__wq_033', 'x__wq_041', 'x__current__netflow_amount_rate_main', 'x__wq_018', 'x__current__reversal_5', 'x__wq_034', 'x__wq_003', 'x__current__D_ROE_4Q', 'x__current__EP_CHINA', 'x__current__D_ROA_4Q', 'x__wq_002', 'x__wq_042', 'x__current__CFOA_LAG', 'x__wq_012', 'x__current__ROE_Q', 'x__current__ROA_Q', 'x__wq_020', 'x__current__OCFP', 'x__current__SP', 'x__current__PROFIT_MARGIN', 'x__current__DPIA', 'x__current__OPA_LAG', 'x__wq_005', 'x__current__ASSET_TURNOVER', 'x__current__NOA', 'x__current__CASH_TO_ASSETS', 'x__current__GPA_TTM', 'x__current__POA', 'x__current__GLA_TTM', 'x__current__RECEIVABLES_GROWTH', 'x__current__OA_CF', 'x__current__INVENTORY_CHANGE_A', 'x__current__gross_profit_rate_ttm', 'x__current__INVENTORY_GROWTH_YOY', 'x__current__NDP', 'x__current__ASSET_GROWTH_YOY', 'x__current__current_ratio_lf']
        FROZEN_E4_FEATURE_COLUMNS = ['x__gtja_070', 'x__current__turn', 'x__wq_016', 'x__wq_013', 'x__gtja_095', 'x__current__volatility_5', 'x__wq_054', 'x__current__atr_pct', 'x__gtja_080', 'x__wq_025', 'x__wq_055', 'x__wq_044', 'x__wq_033', 'x__gtja_043', 'x__wq_041', 'x__current__netflow_amount_rate_main', 'x__wq_018', 'x__gtja_118', 'x__gtja_046', 'x__gtja_003', 'x__current__reversal_5', 'x__wq_034', 'x__gtja_031', 'x__gtja_020', 'x__gtja_040', 'x__wq_003', 'x__gtja_014', 'x__gtja_038', 'x__gtja_015', 'x__gtja_002', 'x__current__D_ROE_4Q', 'x__gtja_112', 'x__current__EP_CHINA', 'x__gtja_008', 'x__current__D_ROA_4Q', 'x__wq_002', 'x__gtja_011', 'x__gtja_110', 'x__gtja_133', 'x__wq_042', 'x__gtja_037', 'x__current__CFOA_LAG', 'x__wq_012', 'x__current__ROE_Q', 'x__current__ROA_Q', 'x__wq_020', 'x__current__OCFP', 'x__current__SP', 'x__current__PROFIT_MARGIN', 'x__current__DPIA', 'x__current__OPA_LAG', 'x__wq_005', 'x__current__ASSET_TURNOVER', 'x__current__NOA', 'x__current__CASH_TO_ASSETS', 'x__current__GPA_TTM', 'x__current__POA', 'x__current__GLA_TTM', 'x__current__RECEIVABLES_GROWTH', 'x__current__OA_CF', 'x__current__INVENTORY_CHANGE_A', 'x__current__gross_profit_rate_ttm', 'x__current__INVENTORY_GROWTH_YOY', 'x__current__NDP', 'x__current__ASSET_GROWTH_YOY', 'x__current__current_ratio_lf']
        SELECTED_FEATURE_COLUMNS = list(E2_BASE_FEATURE_COLUMNS)

        BALANCED_CONFIG = {
            "name": "balanced",
            "learning_rate": 0.020,
            "temperature": 0.50,
            "mse_lambda": 0.03,
            "l1_lambda": 0.0004,
            "l2_lambda": 0.0010,
            "group_lambda": 0.0005,
            "concentration_lambda": 0.20,
            "family_cap": 0.45,
            "anchor_lambda": 0.02,
        }

        DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        DTYPE = torch.float32
        if DEVICE.type == "cuda":
            torch.backends.cuda.matmul.allow_tf32 = True
            try:
                torch.set_float32_matmul_precision("high")
            except Exception:
                pass

        def _cs_zscore(series):
            values = pd.to_numeric(series, errors="coerce")
            if int(values.notna().sum()) == 0:
                return pd.Series(0.0, index=series.index, dtype="float64")
            median = values.median()
            values = values.fillna(0.0 if not np.isfinite(median) else median)
            std = values.std(ddof=0)
            if not np.isfinite(std) or std <= 1e-12:
                return pd.Series(0.0, index=series.index, dtype="float64")
            return (values - values.mean()) / std

        def _query_exposures(query_start, query_end):
            try:
                frame = dai.query(
                    f"SELECT * FROM {EXPOSURE_TABLE}",
                    filters={
                        "date": [
                            _date_string(query_start),
                            _date_string(query_end),
                        ]
                    },
                    compression=True,
                ).df()
            except Exception:
                return pd.DataFrame(columns=["date", "instrument"]), []

            if frame.empty or not {"date", "instrument"}.issubset(frame.columns):
                return pd.DataFrame(columns=["date", "instrument"]), []

            frame["date"] = pd.to_datetime(frame["date"]).dt.normalize()
            frame["instrument"] = frame["instrument"].astype(str)
            frame = frame.drop_duplicates(["date", "instrument"], keep="last")

            numeric = []
            for column in frame.columns:
                if column in {"date", "instrument"}:
                    continue
                converted = pd.to_numeric(frame[column], errors="coerce")
                if converted.notna().mean() >= 0.50 and converted.nunique(dropna=True) > 5:
                    frame[column] = converted
                    numeric.append(column)

            renamed = {column: f"exposure__{column}" for column in numeric}
            frame = frame[["date", "instrument", *numeric]].rename(columns=renamed)
            return frame, list(renamed.values())

        def _selected_registry():
            current_family = {}
            for family, mapping in CURRENT_FACTOR_GROUPS.items():
                for raw_name in mapping:
                    current_family[raw_name] = family

            rows = []
            for x_column in SELECTED_FEATURE_COLUMNS:
                factor_name = x_column[len("x__"):]
                if factor_name.startswith("current__"):
                    raw_name = factor_name[len("current__"):]
                    rows.append({
                        "factor_name": factor_name,
                        "x_column": x_column,
                        "source": "current",
                        "family": current_family[raw_name],
                    })
                elif factor_name.startswith("wq_"):
                    rows.append({
                        "factor_name": factor_name,
                        "x_column": x_column,
                        "source": "worldquant101",
                        "family": WQ_META[factor_name][0],
                    })
                elif factor_name.startswith("gtja_"):
                    rows.append({
                        "factor_name": factor_name,
                        "x_column": x_column,
                        "source": "gtja191",
                        "family": GTJA_META[factor_name][0],
                    })
                else:
                    raise ValueError(f"Unsupported selected factor: {factor_name}")
            return pd.DataFrame(rows)

        def _build_model_range(
            range_start,
            range_end,
            financial_table,
            bar1m_table,
            include_training_columns=False,
        ):
            target_start = _as_timestamp(range_start).normalize()
            target_end = _as_timestamp(range_end).normalize()
            warm_start = target_start - pd.Timedelta(
                days=TECHNICAL_WARMUP_CALENDAR_DAYS
            )

            panel = build_feature_panel(
                start_date=warm_start,
                end_date=target_end,
                factorlib_table=DEFAULT_FACTORLIB_TABLE,
                financial_table=financial_table,
            )
            bars = _query_daily_ohlcv(
                bar1m_table,
                warm_start,
                target_end,
            )

            # Preserve the official instrument universe. Missing market rows become
            # neutral model inputs rather than deleting constituents.
            panel = panel.merge(
                bars,
                on=["date", "instrument"],
                how="left",
                validate="one_to_one",
            )
            panel = panel.sort_values(["instrument", "date"]).reset_index(drop=True)
            panel = compute_worldquant_batch(panel)
            if INCLUDE_GTJA:
                panel = compute_gtja_batch(panel)
            panel = panel.sort_values(["date", "instrument"]).reset_index(drop=True)
            panel = build_literature_composite(panel, mode="category_equal")
            panel = panel.loc[
                panel["date"].between(target_start, target_end)
            ].copy()

            feature_values = {}
            for x_column in SELECTED_FEATURE_COLUMNS:
                factor_name = x_column[len("x__"):]
                if factor_name.startswith("current__"):
                    raw_name = factor_name[len("current__"):]
                    rank_name = f"rank__{raw_name}"
                    if rank_name not in panel.columns:
                        raise KeyError(f"Missing current-factor rank: {rank_name}")
                    values = pd.to_numeric(
                        panel[rank_name], errors="coerce"
                    ) - 0.5
                else:
                    if factor_name not in panel.columns:
                        raise KeyError(f"Missing WorldQuant factor: {factor_name}")
                    values = _cs_rank(
                        pd.to_numeric(panel[factor_name], errors="coerce"),
                        panel["date"],
                    ) - 0.5
                feature_values[x_column] = (
                    pd.to_numeric(values, errors="coerce")
                    .replace([np.inf, -np.inf], np.nan)
                    .fillna(0.0)
                    .astype("float32")
                )

            panel = pd.concat(
                [panel, pd.DataFrame(feature_values, index=panel.index)],
                axis=1,
            )

            if include_training_columns:
                exposures, exposure_columns = _query_exposures(
                    target_start,
                    target_end,
                )
                if exposure_columns:
                    panel = panel.merge(
                        exposures,
                        on=["date", "instrument"],
                        how="left",
                        validate="one_to_one",
                    )
                else:
                    exposure_columns = []
                    for column in FALLBACK_RISK_COLUMNS:
                        if column in panel.columns:
                            target = f"exposure__fallback__{column}"
                            panel[target] = panel[column]
                            exposure_columns.append(target)

                keep = [
                    "date",
                    "instrument",
                    "daily_return",
                    *SELECTED_FEATURE_COLUMNS,
                    *[c for c in PUBLIC_POOL_PROXY_COLUMNS if c in panel.columns],
                    *exposure_columns,
                ]
                result = panel[list(dict.fromkeys(keep))].copy()
                result.attrs["exposure_columns"] = exposure_columns
                return result

            return panel[
                ["date", "instrument", *SELECTED_FEATURE_COLUMNS]
            ].copy()

        def _prepare_training_panel(frame):
            exposure_raw = list(frame.attrs.get("exposure_columns", []))
            frame = frame.sort_values(["date", "instrument"]).reset_index(drop=True)

            pool_features = []
            for source in PUBLIC_POOL_PROXY_COLUMNS:
                if source not in frame.columns:
                    continue
                target = f"pool__{source}"
                frame[target] = (
                    frame.groupby("date")[source]
                    .transform(_cs_zscore)
                    .astype("float32")
                )
                pool_features.append(target)

            coverage = (
                frame[exposure_raw].notna().mean().sort_values(ascending=False)
                if exposure_raw else pd.Series(dtype=float)
            )
            exposure_raw = coverage.loc[coverage.ge(0.80)].index.tolist()[:12]

            risk_features = []
            for source in exposure_raw:
                target = f"risk__{source}"
                frame[target] = (
                    frame.groupby("date")[source]
                    .transform(_cs_zscore)
                    .astype("float32")
                )
                risk_features.append(target)

            if len(pool_features) < 6:
                raise RuntimeError(
                    f"Too few public-pool proxy features: {pool_features}"
                )
            if len(risk_features) < 3:
                # Exposure table is optional; create allowed factorlib risk proxies.
                risk_features = []
                for source in FALLBACK_RISK_COLUMNS:
                    if source not in frame.columns:
                        continue
                    target = f"risk__fallback__{source}"
                    frame[target] = (
                        frame.groupby("date")[source]
                        .transform(_cs_zscore)
                        .astype("float32")
                    )
                    risk_features.append(target)

            if len(risk_features) < 3:
                raise RuntimeError(
                    f"Too few risk-control features: {risk_features}"
                )

            keep = [
                "date",
                "instrument",
                "daily_return",
                *SELECTED_FEATURE_COLUMNS,
                *pool_features,
                *risk_features,
            ]
            return frame[keep].copy(), pool_features, risk_features

        def build_strict_t1_dataset(full_panel, pool_features, risk_features):
            data = full_panel.copy()
            dates = np.sort(data["date"].unique())
            mapping = pd.DataFrame({
                "date": dates[:-1],
                "target_date": dates[1:],
            })
            future = data[["date", "instrument", "daily_return"]].rename(
                columns={
                    "date": "target_date",
                    "daily_return": "target_return",
                }
            )
            model = data.merge(mapping, on="date", how="inner")
            model = model.merge(
                future,
                on=["target_date", "instrument"],
                how="inner",
                validate="one_to_one",
            )
            model["target_return"] = pd.to_numeric(
                model["target_return"], errors="coerce"
            )
            model = model.dropna(subset=["target_return"]).copy()
            model["target_rank"] = (
                model.groupby("date")["target_return"]
                .rank(method="average", pct=True)
                - 0.5
            )
            model["target_z"] = (
                model.groupby("date")["target_rank"]
                .transform(_cs_zscore)
            )
            environment = model.groupby("date")["target_return"].agg(
                market_return="mean",
                cross_sectional_volatility="std",
            )
            model = model.merge(environment, on="date", how="left")
            model = model.replace([np.inf, -np.inf], np.nan)
            required = [
                "target_z",
                "target_return",
                *pool_features,
                *risk_features,
            ]
            model = model.dropna(subset=required).copy()
            return model.sort_values(
                ["date", "instrument"]
            ).reset_index(drop=True)

        def _date_equal_weights(dates):
            dates = pd.Series(pd.to_datetime(dates)).reset_index(drop=True)
            counts = dates.value_counts()
            weights = dates.map(
                lambda value: 1.0 / counts.loc[value]
            ).to_numpy(float)
            weights /= weights.sum()
            return weights

        def _soft_threshold(value, threshold):
            if value > threshold:
                return value - threshold
            if value < -threshold:
                return value + threshold
            return 0.0

        def fit_date_balanced_elastic_net(
            X,
            y,
            dates,
            lam=0.003,
            l1_ratio=0.50,
            max_iter=5000,
            tol=1e-9,
        ):
            X = np.asarray(X, dtype=float)
            y = np.asarray(y, dtype=float)
            weights = _date_equal_weights(dates)
            gram = X.T @ (weights[:, None] * X)
            rhs = X.T @ (weights * y)
            beta = np.zeros(X.shape[1], dtype=float)
            l1 = float(lam) * float(l1_ratio)
            l2 = float(lam) * (1.0 - float(l1_ratio))
            diagonal = np.diag(gram)
            for _ in range(max_iter):
                old = beta.copy()
                for j in range(len(beta)):
                    partial = rhs[j] - (
                        gram[j] @ beta - gram[j, j] * beta[j]
                    )
                    beta[j] = _soft_threshold(
                        partial, l1
                    ) / max(diagonal[j] + l2, 1e-12)
                if np.max(np.abs(beta - old)) < tol:
                    break
            return beta

        def add_rolling_incremental_target(
            model_data,
            pool_features,
            window=60,
            step=20,
            lam=0.003,
            l1_ratio=0.50,
        ):
            frame = model_data.sort_values(
                ["date", "instrument"]
            ).copy()
            dates = np.array(sorted(frame["date"].unique()))
            prediction = pd.Series(
                np.nan, index=frame.index, dtype="float64"
            )
            for start in range(window, len(dates), step):
                train_dates = dates[start - window:start]
                predict_dates = dates[
                    start:min(start + step, len(dates))
                ]
                train = frame.loc[frame["date"].isin(train_dates)]
                test = frame.loc[frame["date"].isin(predict_dates)]
                beta = fit_date_balanced_elastic_net(
                    train[pool_features].to_numpy(float),
                    train["target_z"].to_numpy(float),
                    train["date"],
                    lam=lam,
                    l1_ratio=l1_ratio,
                )
                prediction.loc[test.index] = (
                    test[pool_features].to_numpy(float) @ beta
                )
            frame["pool_prediction"] = prediction
            frame["incremental_target"] = (
                frame["target_z"] - frame["pool_prediction"]
            )
            valid = frame["incremental_target"].notna()
            frame.loc[valid, "incremental_target_z"] = (
                frame.loc[valid]
                .groupby("date")["incremental_target"]
                .transform(_cs_zscore)
            )
            return frame.dropna(
                subset=["incremental_target_z"]
            ).reset_index(drop=True)


        def _select_gtja_extension(model_data, selection_end, keep_count=20):
            """Select 20 GTJA additions without rerunning the full research zoo.

            The E2 architecture stays frozen. GTJA candidates are ranked by the
            absolute daily ICIR on the pre-2024 selection sample and de-duplicated
            at a 0.98 sampled cross-sectional correlation threshold.
            """
            sample = model_data.loc[
                model_data["date"].le(_as_timestamp(selection_end).normalize())
            ].copy()
            if sample.empty:
                raise RuntimeError("GTJA selection sample is empty.")

            # The model inputs are already daily percentile ranks and target_z is
            # a monotone standardization of the daily target rank. Therefore the
            # daily Pearson correlation below is the same ranking signal used by
            # the research Spearman screen, but it is computed for all 26 GTJA
            # candidates in one vectorized pass.
            columns = list(ALL_GTJA_FEATURE_COLUMNS)
            work = sample[["date", "target_z", *columns]].copy()
            numeric = ["target_z", *columns]
            work[numeric] = work[numeric].astype("float32")
            centered = work[numeric] - work.groupby("date")[numeric].transform("mean")
            y = centered["target_z"]
            x = centered[columns]
            numerator = x.mul(y, axis=0).groupby(work["date"]).sum()
            x_ss = x.pow(2).groupby(work["date"]).sum()
            y_ss = y.pow(2).groupby(work["date"]).sum()
            denominator = np.sqrt(x_ss.mul(y_ss, axis=0))
            daily_corr = numerator.div(denominator).replace([np.inf, -np.inf], np.nan)
            ic_mean = daily_corr.mean(axis=0).fillna(0.0)
            ic_std = daily_corr.std(axis=0, ddof=1).replace(0.0, np.nan)
            ic_ir = ic_mean.div(ic_std).replace([np.inf, -np.inf], np.nan).fillna(0.0)

            ranking = pd.DataFrame({
                "x_column": columns,
                "priority": ic_ir.abs().reindex(columns).to_numpy(),
                "ic_mean": ic_mean.reindex(columns).to_numpy(),
                "ic_ir": ic_ir.reindex(columns).to_numpy(),
            }).sort_values(
                ["priority", "x_column"], ascending=[False, True]
            )
            del work, centered, x, y, numerator, x_ss, y_ss, denominator, daily_corr

            dates = np.array(sorted(sample["date"].unique()))
            stride = max(1, len(dates) // 120)
            sampled_dates = dates[::stride]
            corr_columns = [
                *E2_BASE_FEATURE_COLUMNS,
                *ALL_GTJA_FEATURE_COLUMNS,
            ]
            correlation = (
                sample.loc[sample["date"].isin(sampled_dates), corr_columns]
                .astype("float32")
                .corr()
                .abs()
            )

            kept = []
            reference = list(E2_BASE_FEATURE_COLUMNS)
            for column in ranking["x_column"]:
                old_columns = reference + kept
                if old_columns:
                    max_corr = correlation.loc[column, old_columns].max()
                    if pd.notna(max_corr) and float(max_corr) > 0.98:
                        continue
                kept.append(column)
                if len(kept) >= keep_count:
                    break

            # Preserve the intended 46 + 20 architecture even if the correlation
            # gate is unusually restrictive in a future execution environment.
            if len(kept) < keep_count:
                for column in ranking["x_column"]:
                    if column not in kept:
                        kept.append(column)
                    if len(kept) >= keep_count:
                        break

            if len(kept) != keep_count:
                raise RuntimeError(
                    f"Expected {keep_count} GTJA extensions, got {len(kept)}."
                )
            return kept

        def stress_thresholds(train_frame):
            daily = train_frame.groupby("date")[[
                "market_return",
                "cross_sectional_volatility",
            ]].first()
            return {
                "market_q20": float(daily["market_return"].quantile(0.20)),
                "vol_q80": float(
                    daily["cross_sectional_volatility"].quantile(0.80)
                ),
            }

        class GroupCompetitionRanker(nn.Module):
            def __init__(self, n_features, initial_beta=None):
                super().__init__()
                if initial_beta is None:
                    initial_beta = np.random.normal(
                        0.0, 0.01, size=n_features
                    )
                self.beta = nn.Parameter(
                    torch.as_tensor(initial_beta, dtype=DTYPE)
                )

            def forward(self, x):
                return x @ self.beta

        def _m1_anchor(selected_registry):
            beta = np.zeros(len(selected_registry), dtype=np.float32)
            current = selected_registry.loc[
                selected_registry["source"].eq("current")
            ]
            for _, block in current.groupby("family"):
                indices = block.index.to_numpy(int)
                if len(indices):
                    beta[indices] = 0.25 / len(indices)
            return beta

        def _date_blocks(frame, feature_columns, risk_columns):
            blocks = []
            ordered = frame.sort_values(["date", "instrument"])
            cache_on_device = DEVICE.type == "cuda"
            for date, block in ordered.groupby("date", sort=True):
                item = {
                    "date": pd.Timestamp(date),
                    "x": block[feature_columns].to_numpy(np.float32),
                    "risk": block[risk_columns].to_numpy(np.float32),
                    "target_z": block["target_z"].to_numpy(np.float32),
                    "incremental_z": block[
                        "incremental_target_z"
                    ].to_numpy(np.float32),
                    "target_return": block[
                        "target_return"
                    ].to_numpy(np.float32),
                    "market_return": float(
                        block["market_return"].iloc[0]
                    ),
                    "cross_sectional_volatility": float(
                        block["cross_sectional_volatility"].iloc[0]
                    ),
                }
                if cache_on_device:
                    for key in (
                        "x",
                        "risk",
                        "target_z",
                        "incremental_z",
                        "target_return",
                    ):
                        item[key] = torch.as_tensor(
                            item[key],
                            dtype=DTYPE,
                            device=DEVICE,
                        )
                blocks.append(item)
            return blocks

        def _device_tensor(value):
            if torch.is_tensor(value):
                return value
            return torch.as_tensor(
                value, dtype=DTYPE, device=DEVICE
            )

        def _neutralize_torch(score, risk, ridge=1e-4):
            R = torch.cat(
                [
                    torch.ones(
                        (risk.shape[0], 1),
                        dtype=score.dtype,
                        device=score.device,
                    ),
                    risk,
                ],
                dim=1,
            )
            eye = torch.eye(
                R.shape[1],
                dtype=score.dtype,
                device=score.device,
            )
            system = R.T @ R + ridge * eye
            coef = torch.linalg.solve(system, R.T @ score)
            residual = score - R @ coef
            return (
                residual - residual.mean()
            ) / (residual.std(unbiased=False) + 1e-6)

        def _ratio_torch(values):
            values = torch.stack(values)
            if values.numel() < 2:
                return values.mean() * 0.0
            return values.mean() / (
                values.std(unbiased=True) + 1e-6
            )

        def competition_loss(
            model,
            blocks,
            thresholds,
            selected_registry,
            config,
            anchor,
        ):
            daily_ic = []
            daily_inc = []
            daily_spread = []
            stress_ic = []
            mse_terms = []
            temperature = float(config["temperature"])

            for block in blocks:
                x = _device_tensor(block["x"])
                risk = _device_tensor(block["risk"])
                target = _device_tensor(block["target_z"])
                incremental = _device_tensor(block["incremental_z"])
                returns = _device_tensor(block["target_return"])

                raw = model(x)
                neutral = _neutralize_torch(raw, risk)
                daily_ic.append(torch.mean(neutral * target))
                daily_inc.append(torch.mean(neutral * incremental))

                long_weight = torch.softmax(
                    neutral / temperature, dim=0
                )
                short_weight = torch.softmax(
                    -neutral / temperature, dim=0
                )
                daily_spread.append(
                    torch.sum(
                        (long_weight - short_weight) * returns
                    )
                )
                mse_terms.append(torch.mean((raw - target) ** 2))

                if (
                    block["market_return"]
                    <= thresholds["market_q20"]
                    or block["cross_sectional_volatility"]
                    >= thresholds["vol_q80"]
                ):
                    stress_ic.append(daily_ic[-1])

            ic_mean = torch.stack(daily_ic).mean()
            ic_ir = _ratio_torch(daily_ic)
            spread_sharpe = _ratio_torch(daily_spread)
            stress_icir = _ratio_torch(
                stress_ic if stress_ic else daily_ic
            )
            incremental_mean = torch.stack(daily_inc).mean()
            incremental_ir = _ratio_torch(daily_inc)

            def unit(value, scale):
                return 0.5 + 0.5 * torch.tanh(value / scale)

            A = 0.25 * (
                unit(ic_mean, 0.010)
                + unit(ic_ir, 0.20)
                + unit(spread_sharpe, 0.80)
                + unit(stress_icir, 0.20)
            )
            B = 0.50 * (
                unit(incremental_mean, 0.005)
                + unit(incremental_ir, 0.10)
            )
            proxy = 0.30 * A + 0.70 * B

            beta = model.beta
            l1 = torch.sum(torch.abs(beta))
            l2 = torch.sum(beta ** 2)

            group_terms = []
            family_mass = []
            for _, group in selected_registry.groupby(
                "family", sort=False
            ):
                idx = torch.as_tensor(
                    group.index.to_numpy(int),
                    dtype=torch.long,
                    device=DEVICE,
                )
                group_beta = beta[idx]
                group_terms.append(
                    math.sqrt(len(group))
                    * torch.linalg.vector_norm(group_beta, ord=2)
                )
                family_mass.append(torch.sum(torch.abs(group_beta)))

            group_penalty = torch.stack(group_terms).sum()
            masses = torch.stack(family_mass)
            shares = masses / (masses.sum() + 1e-8)
            concentration = torch.sum(
                torch.relu(
                    shares - float(config["family_cap"])
                ) ** 2
            )

            anchor_tensor = torch.as_tensor(
                anchor, dtype=DTYPE, device=DEVICE
            )
            anchor_penalty = torch.mean(
                (beta - anchor_tensor) ** 2
            )
            mse = torch.stack(mse_terms).mean()

            return (
                -proxy
                + float(config["mse_lambda"]) * mse
                + float(config["l1_lambda"]) * l1
                + float(config["l2_lambda"]) * l2
                + float(config["group_lambda"]) * group_penalty
                + float(config["concentration_lambda"])
                * concentration
                + float(config["anchor_lambda"]) * anchor_penalty
            )

        def fit_group_fixed_epochs(
            train_frame,
            selected_registry,
            risk_columns,
            config,
            epochs,
            seed=17,
            batch_dates=36,
        ):
            random.seed(seed)
            np.random.seed(seed)
            torch.manual_seed(seed)
            if DEVICE.type == "cuda":
                torch.cuda.manual_seed_all(seed)

            selected_registry = selected_registry.reset_index(drop=True)
            features = selected_registry["x_column"].tolist()
            anchor = _m1_anchor(selected_registry)

            model = GroupCompetitionRanker(
                len(features),
                initial_beta=anchor,
            ).to(DEVICE)
            optimizer = torch.optim.Adam(
                model.parameters(),
                lr=float(config["learning_rate"]),
            )
            blocks = _date_blocks(
                train_frame,
                features,
                risk_columns,
            )
            thresholds = stress_thresholds(train_frame)

            for _ in range(max(1, int(epochs))):
                selected_blocks = random.sample(
                    blocks,
                    k=min(batch_dates, len(blocks)),
                )
                optimizer.zero_grad(set_to_none=True)
                loss = competition_loss(
                    model,
                    selected_blocks,
                    thresholds,
                    selected_registry,
                    config,
                    anchor,
                )
                if not torch.isfinite(loss):
                    raise RuntimeError(
                        "Non-finite Group Competition loss."
                    )
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(), 5.0
                )
                optimizer.step()

            beta = (
                model.beta.detach().cpu().numpy().astype("float32")
            )
            return beta
        # ============================================================

        requested_start = _as_timestamp(block_start).normalize()
        requested_end = _as_timestamp(block_end).normalize()
        SELECTED_FEATURE_COLUMNS = list(FROZEN_E4_FEATURE_COLUMNS)
        raw_panel = _build_model_range(
            requested_start,
            requested_end,
            financial_table=datasources["financial"],
            bar1m_table=datasources["bar1m"],
            include_training_columns=True,
        )
        prepared_panel, _, _ = _prepare_training_panel(raw_panel)
        prepared_panel["date"] = pd.to_datetime(prepared_panel["date"]).dt.normalize()
        prepared_panel["instrument"] = prepared_panel["instrument"].astype(str)
        missing = [column for column in E4_FEATURES if column not in prepared_panel.columns]
        if missing:
            raise KeyError(f"E4 block missing columns: {missing[:20]}")
        return (
            prepared_panel.loc[
                prepared_panel["date"].between(requested_start, requested_end),
                ["date", "instrument", *E4_FEATURES],
            ]
            .drop_duplicates(["date", "instrument"], keep="last")
            .sort_values(["date", "instrument"])
            .reset_index(drop=True)
        )
    def _build_phase1_features(bar_table, block_start, block_end):
        datasources = {"bar1m": bar_table}
        start_date = block_start
        end_date = block_end
        """
        BigAlpha 2026 microstructure XGBoost production submission.

        Contract:
          - the training interval is fixed in code and never depends on the injected
            test start/end dates;
          - fixed public training data are read from bigalpha_2026_stock_bar1m;
          - the injected datasources["bar1m"] table is used only for test inference;
          - feature set and XGBoost hyperparameters are frozen;
          - output contains exactly date / instrument / factor and preserves the
            complete historical CSI-1000 universe with neutral fallback values.
        """
        import gc
        import os
        import warnings

        import numpy as np
        import pandas as pd
        import dai
        import xgboost as xgb

        if not isinstance(datasources, dict):
            raise TypeError("datasources must be a dictionary supplied by the platform.")
        if "bar1m" not in datasources:
            raise KeyError('datasources missing required logical source "bar1m".')

        # Fixed public training contract.  These dates never depend on the injected
        # test interval, which prevents test-period leakage and satisfies the
        # train-inside-main submission pattern used by the official examples.
        TRAIN_BAR1M_TABLE = "bigalpha_2026_stock_bar1m"
        TEST_BAR1M_TABLE = datasources["bar1m"]
        INSTRUMENT_TABLE = "bigalpha_2026_instruments"
        TRAIN_START = pd.Timestamp("2021-01-01")
        TRAIN_END = pd.Timestamp("2023-12-31")

        MIN_VALID_MINUTES = 180
        RANDOM_STATE = 2026
        EPS = 1e-8

        # Frozen Phase-1 production model.  No validation search or early stopping
        # is performed during submission.
        XGB_MAX_DEPTH = 3
        XGB_N_ESTIMATORS = 400
        XGB_LEARNING_RATE = 0.03
        XGB_THREADS = min(16, os.cpu_count() or 4)

        CORE_CHANNELS = [
            "spread_bps", "log_depth_1", "log_depth_5", "top_depth_share",
            "depth_per_spread", "obi_1", "obi_5", "weighted_imbalance",
            "near_far_obi", "official_pressure", "signed_pressure",
            "linear_pressure", "microprice_deviation", "abs_mid_return",
            "trade_to_depth", "amihud_intraday", "fragility",
            "alignment_obi_return", "response_ratio", "absorption_score",
        ]
        TAIL_CHANNELS = [
            "spread_bps", "log_depth_5", "obi_1", "obi_5",
            "weighted_imbalance", "official_pressure", "signed_pressure",
            "microprice_deviation", "trade_to_depth", "fragility",
            "absorption_score",
        ]
        SELECTED_FEATURES = [
            "depth_per_spread__std",
            "obi_1__last_z",
            "linear_pressure__mean",
            "abs_mid_return__std",
            "absorption_score__mean",
            "fragility__std",
            "signed_pressure__last_z",
            "trade_to_depth__std",
            "fragility__close30_minus_full",
            "alignment_obi_return__mean",
            "official_pressure__std",
            "log_depth_5__close30_minus_full",
            "fragility__last_z",
            "obi_5__std",
            "log_depth_1__std",
            "log_depth_5__last_z",
            "log_depth_5__std",
            "top_depth_share__std",
            "absorption_score__last_z",
            "official_pressure__last_z",
            "near_far_obi__std",
            "obi_1__std",
            "weighted_imbalance__last_z",
            "obi_1__close30_minus_full",
            "response_ratio__mean",
        ]

        def _timestamp_string(value):
            timestamp = pd.Timestamp(value)
            if timestamp.tzinfo is not None:
                timestamp = timestamp.tz_localize(None)
            return timestamp.strftime("%Y-%m-%d %H:%M:%S")

        def _month_intervals(start_date, end_date):
            start = pd.Timestamp(start_date)
            end = pd.Timestamp(end_date)
            if start.tzinfo is not None:
                start = start.tz_localize(None)
            if end.tzinfo is not None:
                end = end.tz_localize(None)
            if end == end.normalize():
                end = end + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
            current = start.normalize().replace(day=1)
            while current <= end:
                next_month = current + pd.offsets.MonthBegin(1)
                yield max(start, current), min(end, next_month - pd.Timedelta(seconds=1))
                current = next_month

        def _feature_columns():
            columns = []
            for channel in CORE_CHANNELS:
                columns.extend([f"{channel}__mean", f"{channel}__std"])
            for channel in TAIL_CHANNELS:
                columns.extend([
                    f"{channel}__close30_minus_full",
                    f"{channel}__last_z",
                ])
            columns.append("official_hf_factor")
            return columns

        def build_daily_feature_sql(bar1m_table):
            mean_std_items = []
            last_items = []
            close30_items = []
            final_tail_items = []

            for channel in CORE_CHANNELS:
                mean_std_items.extend([
                    f"AVG({channel}) AS {channel}__mean",
                    f"NANSTD({channel}) AS {channel}__std",
                ])

            for channel in TAIL_CHANNELS:
                last_items.append(f"LAST({channel} ORDER BY date) AS {channel}__last")
                close30_items.append(
                    f"AVG(CASE WHEN STRFTIME(date, '%H:%M:%S') >= '14:30:00' "
                    f"THEN {channel} ELSE NULL END) AS {channel}__close30_mean"
                )
                final_tail_items.extend([
                    f"{channel}__close30_mean - {channel}__mean "
                    f"AS {channel}__close30_minus_full",
                    f"CASE WHEN {channel}__std > 0 THEN "
                    f"({channel}__last - {channel}__mean) / {channel}__std "
                    f"ELSE NULL END AS {channel}__last_z",
                ])

            aggregate_sql = ",\n            ".join([
                "CAST(trading_day AS DATETIME) AS date",
                "instrument",
                "COUNT(*) AS minute_count",
                "LAST(close_float ORDER BY date) AS daily_close",
                *mean_std_items,
                *last_items,
                *close30_items,
                "QUANTILE(abs_mid_return, 0.8) AS volatility_threshold",
                "NANSTD(CASE WHEN mid_price > 0 AND prev_mid_price > 0 "
                "THEN LOG(mid_price / prev_mid_price) ELSE NULL END) AS volatility",
            ])
            final_tail_sql = ",\n            ".join(final_tail_items)
            mean_std_select = ",\n            ".join(
                f"{channel}__mean, {channel}__std" for channel in CORE_CHANNELS
            )

            return f"""
            WITH cte_bar1m AS (
                SELECT
                    date,
                    instrument,
                    close * 1.0 AS close_float,
                    volume * 1.0 AS volume_float,
                    amount * 1.0 AS amount_float,
                    STRFTIME(date, '%Y-%m-%d') AS trading_day,

                    ask_price1 * 1.0 AS ask_p1,
                    bid_price1 * 1.0 AS bid_p1,
                    (ask_price1 + bid_price1) / 2.0 AS mid_price,
                    (ask_price1 - bid_price1)
                        / (((ask_price1 + bid_price1) / 2.0) + {EPS}) AS relative_spread,

                    COALESCE(bid_volume1, 0) * 1.0 AS bid_v1,
                    COALESCE(bid_volume2, 0) * 1.0 AS bid_v2,
                    COALESCE(bid_volume3, 0) * 1.0 AS bid_v3,
                    COALESCE(bid_volume4, 0) * 1.0 AS bid_v4,
                    COALESCE(bid_volume5, 0) * 1.0 AS bid_v5,
                    COALESCE(ask_volume1, 0) * 1.0 AS ask_v1,
                    COALESCE(ask_volume2, 0) * 1.0 AS ask_v2,
                    COALESCE(ask_volume3, 0) * 1.0 AS ask_v3,
                    COALESCE(ask_volume4, 0) * 1.0 AS ask_v4,
                    COALESCE(ask_volume5, 0) * 1.0 AS ask_v5,

                    (
                        COALESCE(bid_volume1, 0) * 1.0
                        + COALESCE(bid_volume2, 0) * 1.0
                        + COALESCE(bid_volume3, 0) * 1.0
                        + COALESCE(bid_volume4, 0) * 1.0
                        + COALESCE(bid_volume5, 0) * 1.0
                    ) AS bid_depth_5,
                    (
                        COALESCE(ask_volume1, 0) * 1.0
                        + COALESCE(ask_volume2, 0) * 1.0
                        + COALESCE(ask_volume3, 0) * 1.0
                        + COALESCE(ask_volume4, 0) * 1.0
                        + COALESCE(ask_volume5, 0) * 1.0
                    ) AS ask_depth_5,

                    (
                        COALESCE(bid_volume1, 0) * 1.0
                        + COALESCE(bid_volume2, 0) * EXP(-0.3)
                        + COALESCE(bid_volume3, 0) * EXP(-0.6)
                        + COALESCE(bid_volume4, 0) * EXP(-0.9)
                        + COALESCE(bid_volume5, 0) * EXP(-1.2)
                    ) AS weight_bid,
                    (
                        COALESCE(ask_volume1, 0) * 1.0
                        + COALESCE(ask_volume2, 0) * EXP(-0.3)
                        + COALESCE(ask_volume3, 0) * EXP(-0.6)
                        + COALESCE(ask_volume4, 0) * EXP(-0.9)
                        + COALESCE(ask_volume5, 0) * EXP(-1.2)
                    ) AS weight_ask,

                    (COALESCE(bid_volume1, 0) * 1.0
                        + COALESCE(bid_volume2, 0) * 1.0) AS near_bid,
                    (COALESCE(ask_volume1, 0) * 1.0
                        + COALESCE(ask_volume2, 0) * 1.0) AS near_ask,
                    (COALESCE(bid_volume4, 0) * 1.0
                        + COALESCE(bid_volume5, 0) * 1.0) AS far_bid,
                    (COALESCE(ask_volume4, 0) * 1.0
                        + COALESCE(ask_volume5, 0) * 1.0) AS far_ask

                FROM {bar1m_table}
                WHERE ask_price1 > 0
                  AND bid_price1 > 0
                  AND ask_price1 >= bid_price1
            ),

            cte_static AS (
                SELECT
                    *,
                    relative_spread * 10000.0 AS spread_bps,
                    LOG(1.0 + bid_v1 + ask_v1) AS log_depth_1,
                    LOG(1.0 + bid_depth_5 + ask_depth_5) AS log_depth_5,
                    (bid_v1 + ask_v1)
                        / (bid_depth_5 + ask_depth_5 + {EPS}) AS top_depth_share,
                    LOG(1.0 + bid_depth_5 + ask_depth_5)
                        / (relative_spread * 10000.0 + 1.0) AS depth_per_spread,
                    (bid_v1 - ask_v1) / (bid_v1 + ask_v1 + {EPS}) AS obi_1,
                    (bid_depth_5 - ask_depth_5)
                        / (bid_depth_5 + ask_depth_5 + {EPS}) AS obi_5,
                    (weight_bid - weight_ask)
                        / (weight_bid + weight_ask + {EPS}) AS weighted_imbalance,
                    (
                        (near_bid - near_ask) / (near_bid + near_ask + {EPS})
                        - (far_bid - far_ask) / (far_bid + far_ask + {EPS})
                    ) AS near_far_obi,
                    CASE
                        WHEN bid_v1 + ask_v1 > 0 AND mid_price > 0
                        THEN ((ask_p1 * bid_v1 + bid_p1 * ask_v1)
                            / (bid_v1 + ask_v1) / mid_price - 1.0) * 10000.0
                        ELSE NULL
                    END AS microprice_deviation
                FROM cte_bar1m
            ),

            cte_pressure AS (
                SELECT
                    *,
                    weighted_imbalance * weighted_imbalance
                        / (SQRT(ABS(relative_spread)) + {EPS}) AS official_pressure,
                    weighted_imbalance * ABS(weighted_imbalance)
                        / (SQRT(ABS(relative_spread)) + {EPS}) AS signed_pressure,
                    weighted_imbalance
                        / (SQRT(ABS(relative_spread)) + {EPS}) AS linear_pressure
                FROM cte_static
            ),

            cte_rolling AS (
                SELECT
                    *,
                    LAG(mid_price, 1) OVER (
                        PARTITION BY instrument, trading_day ORDER BY date
                    ) AS prev_mid_price,
                    LAG(weighted_imbalance, 1) OVER (
                        PARTITION BY instrument, trading_day ORDER BY date
                    ) AS prev_weighted_imbalance
                FROM cte_pressure
            ),

            cte_channels AS (
                SELECT
                    *,
                    CASE WHEN mid_price > 0 AND prev_mid_price > 0
                        THEN ABS(mid_price / prev_mid_price - 1.0)
                        ELSE NULL END AS abs_mid_return,
                    volume_float / (bid_depth_5 + ask_depth_5 + {EPS}) AS trade_to_depth,
                    CASE WHEN mid_price > 0 AND prev_mid_price > 0
                        THEN ABS(mid_price / prev_mid_price - 1.0) / (amount_float + 1.0)
                        ELSE NULL END AS amihud_intraday,
                    CASE WHEN mid_price > 0 AND prev_mid_price > 0
                        THEN ABS(mid_price / prev_mid_price - 1.0)
                            * (1.0 + relative_spread * 10000.0)
                            / (LOG(1.0 + bid_depth_5 + ask_depth_5) + {EPS})
                        ELSE NULL END AS fragility,
                    CASE WHEN mid_price > 0 AND prev_mid_price > 0
                        THEN prev_weighted_imbalance * (mid_price / prev_mid_price - 1.0)
                        ELSE NULL END AS alignment_obi_return,
                    CASE
                        WHEN mid_price <= 0 OR prev_mid_price <= 0
                            OR prev_weighted_imbalance IS NULL THEN NULL
                        WHEN prev_weighted_imbalance >= 0
                            THEN (mid_price / prev_mid_price - 1.0)
                                / (ABS(prev_weighted_imbalance) + 0.05)
                        ELSE -(mid_price / prev_mid_price - 1.0)
                                / (ABS(prev_weighted_imbalance) + 0.05)
                    END AS response_ratio,
                    CASE WHEN mid_price > 0 AND prev_mid_price > 0
                            AND prev_weighted_imbalance IS NOT NULL
                        THEN LOG(1.0 + ABS(prev_weighted_imbalance)
                            / (ABS(mid_price / prev_mid_price - 1.0) + 1e-6))
                        ELSE NULL END AS absorption_score
                FROM cte_rolling
            ),

            cte_daily_raw AS (
                SELECT
                    {aggregate_sql}
                FROM cte_channels
                GROUP BY instrument, trading_day
            ),

            cte_daily_features AS (
                SELECT
                    date,
                    instrument,
                    minute_count,
                    daily_close,
                    volatility_threshold,
                    volatility,
                    {mean_std_select},
                    {final_tail_sql},
                    CASE
                        WHEN official_pressure__std > 0
                        THEN -TANH(
                            CASE WHEN volatility > volatility_threshold
                                THEN (official_pressure__last - official_pressure__mean)
                                    / official_pressure__std * 0.7
                                ELSE (official_pressure__last - official_pressure__mean)
                                    / official_pressure__std
                            END
                        )
                        ELSE 0.0
                    END AS official_hf_factor
                FROM cte_daily_raw
            )

            SELECT *
            FROM cte_daily_features
            WHERE minute_count >= {MIN_VALID_MINUTES}
            ORDER BY date, instrument
            """

        def query_daily_features(bar1m_table, start_date, end_date):
            import dai

            sql = build_daily_feature_sql(bar1m_table)
            frame = dai.query(
                sql,
                filters={"date": [_timestamp_string(start_date), _timestamp_string(end_date)]},
                compression=True,
            ).df()
            if frame.empty:
                raise RuntimeError("Daily microstructure feature query returned no rows.")

            frame["date"] = pd.to_datetime(frame["date"]).dt.normalize()
            frame["instrument"] = frame["instrument"].astype(str)
            frame = frame.drop_duplicates(["date", "instrument"], keep="last")

            required = [
                "date", "instrument", "minute_count", "daily_close",
                "volatility_threshold", "volatility", *FEATURE_COLUMNS,
            ]
            missing = [column for column in required if column not in frame.columns]
            if missing:
                raise RuntimeError(f"DAI output is missing required columns: {missing}")

            numeric = [column for column in frame.columns if column not in ("date", "instrument")]
            for column in numeric:
                frame[column] = pd.to_numeric(frame[column], errors="coerce")
            frame = frame.replace([np.inf, -np.inf], np.nan)

            if frame.duplicated(["date", "instrument"]).any():
                raise RuntimeError("Duplicate date-instrument rows after DAI aggregation.")
            if (frame["minute_count"] < MIN_VALID_MINUTES).any():
                raise RuntimeError("Rows below the minimum-minute threshold survived.")
            if frame["daily_close"].isna().all():
                raise RuntimeError("daily_close is entirely missing.")

            return frame.sort_values(["date", "instrument"]).reset_index(drop=True)

        def query_historical_pool(start_date, end_date):
            import dai

            pool = dai.query(
                f"SELECT date, instrument FROM {INSTRUMENT_TABLE}",
                filters={"date": [_timestamp_string(start_date), _timestamp_string(end_date)]},
                compression=True,
            ).df()
            if pool.empty:
                raise RuntimeError("Historical CSI 1000 universe query returned no rows.")
            pool["date"] = pd.to_datetime(pool["date"]).dt.normalize()
            pool["instrument"] = pool["instrument"].astype(str)
            return pool.drop_duplicates(["date", "instrument"]).sort_values(
                ["date", "instrument"]
            ).reset_index(drop=True)

        def attach_next_day_label(all_stock_daily):
            frame = all_stock_daily.sort_values(["instrument", "date"]).copy()
            calendar = pd.Index(sorted(frame["date"].dropna().unique()))
            next_date_map = {
                calendar[index]: calendar[index + 1]
                for index in range(len(calendar) - 1)
            }
            grouped = frame.groupby("instrument", sort=False)
            frame["next_observed_date"] = grouped["date"].shift(-1)
            frame["next_close"] = grouped["daily_close"].shift(-1)
            frame["expected_next_date"] = frame["date"].map(next_date_map)
            valid = (
                frame["next_observed_date"].eq(frame["expected_next_date"])
                & (frame["daily_close"] > 0)
                & (frame["next_close"] > 0)
            )
            frame["forward_return_1d"] = np.where(
                valid,
                frame["next_close"] / frame["daily_close"] - 1.0,
                np.nan,
            )
            frame = frame.drop(
                columns=["next_observed_date", "next_close", "expected_next_date"]
            )
            return frame.sort_values(["date", "instrument"]).reset_index(drop=True)

        def cross_sectional_winsor_zscore(frame, feature_columns, lower=0.01, upper=0.99):
            output = frame[["date", "instrument"]].copy()
            values = frame[list(feature_columns)].apply(
                pd.to_numeric, errors="coerce"
            ).to_numpy(dtype=np.float64)
            result = np.full(values.shape, np.nan, dtype=np.float32)
            date_values = frame["date"].to_numpy()

            # Sort once and slice contiguous date blocks.  The previous implementation
            # scanned the entire date vector once per trading day (O(D*N)).
            order = np.argsort(date_values, kind="stable")
            sorted_dates = date_values[order]
            starts = np.flatnonzero(
                np.r_[True, sorted_dates[1:] != sorted_dates[:-1]]
            )
            ends = np.r_[starts[1:], len(order)]

            for start, end in zip(starts, ends):
                rows = order[start:end]
                block = values[rows]
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", RuntimeWarning)
                    lo = np.nanquantile(block, lower, axis=0)
                    hi = np.nanquantile(block, upper, axis=0)
                    block = np.clip(block, lo, hi)
                    mean = np.nanmean(block, axis=0)
                    std = np.nanstd(block, axis=0)
                block = (block - mean) / np.where(std > 1e-12, std, np.nan)
                result[rows] = block.astype(np.float32)

            output[list(feature_columns)] = result
            return output


        FEATURE_COLUMNS = _feature_columns()

        def _end_of_day(value):
            timestamp = pd.Timestamp(value)
            if timestamp.tzinfo is not None:
                timestamp = timestamp.tz_localize(None)
            return timestamp.normalize() + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)

        def _query_feature_range(table, query_start, query_end):
            pieces = []
            for month_start, month_end in _month_intervals(query_start, query_end):
                try:
                    piece = query_daily_features(table, month_start, month_end)
                except RuntimeError as exc:
                    if "returned no rows" in str(exc):
                        continue
                    raise
                pieces.append(piece)
            if not pieces:
                raise RuntimeError("No monthly feature pieces were produced.")
            frame = pd.concat(pieces, ignore_index=True)
            frame = frame.drop_duplicates(["date", "instrument"], keep="last")
            return frame.sort_values(["date", "instrument"]).reset_index(drop=True)

        def _make_fixed_model():
            return xgb.XGBRegressor(
                objective="reg:squarederror",
                eval_metric="rmse",
                n_estimators=XGB_N_ESTIMATORS,
                max_depth=XGB_MAX_DEPTH,
                learning_rate=XGB_LEARNING_RATE,
                subsample=0.8,
                colsample_bytree=0.8,
                min_child_weight=50.0,
                reg_alpha=0.1,
                reg_lambda=10.0,
                gamma=0.0,
                tree_method="hist",
                predictor="cpu_predictor",
                random_state=RANDOM_STATE,
                n_jobs=XGB_THREADS,
            )

        evaluation_start = pd.Timestamp(start_date)
        evaluation_end = pd.Timestamp(end_date)
        if evaluation_start.tzinfo is not None:
            evaluation_start = evaluation_start.tz_localize(None)
        if evaluation_end.tzinfo is not None:
            evaluation_end = evaluation_end.tz_localize(None)
        evaluation_start_day = evaluation_start.normalize()
        evaluation_end_day = evaluation_end.normalize()
        if evaluation_end_day < evaluation_start_day:
            raise ValueError("end_date is earlier than start_date.")

        # 1) Rebuild the fixed 2021-2023 training panel and next-day labels.
        all_train_daily = _query_feature_range(
            TRAIN_BAR1M_TABLE,
            TRAIN_START,
            _end_of_day(TRAIN_END),
        )
        labelled = attach_next_day_label(all_train_daily)

        query_start = _as_timestamp(block_start).normalize()
        query_end = _end_of_day(block_end)
        frame = query_daily_features(bar_table, query_start, query_end)
        if frame.empty:
            raise RuntimeError("Phase-1 feature query returned no rows.")
        frame["date"] = pd.to_datetime(frame["date"]).dt.normalize()
        frame["instrument"] = frame["instrument"].astype(str)
        missing = [column for column in PHASE1_FEATURES if column not in frame.columns]
        if missing:
            raise KeyError(f"Phase-1 block missing columns: {missing[:20]}")
        return (
            frame.loc[
                frame["date"].between(query_start, _as_timestamp(block_end).normalize()),
                ["date", "instrument", *PHASE1_FEATURES],
            ]
            .drop_duplicates(["date", "instrument"], keep="last")
            .sort_values(["date", "instrument"])
            .reset_index(drop=True)
        )
    def _build_phase3_features(bar_table, block_start, block_end):
        from pathlib import Path

        import gc

        import json

        import math

        import os

        import pickle

        import time

        import warnings

        import numpy as np

        import pandas as pd

        from sklearn.linear_model import ElasticNet

        BOOK_LEVELS = (1, 2, 3, 4, 5)

        MIN_VALID_MINUTES = 180

        SHOCK_PERCENTILE = 0.90

        EPS = 1e-8

        def _timestamp_string(value) -> str:
            ts = pd.Timestamp(value)
            if ts.tzinfo is not None:
                ts = ts.tz_localize(None)
            return ts.strftime("%Y-%m-%d %H:%M:%S")

        def _month_intervals(start_date, end_date):
            start = pd.Timestamp(start_date).normalize()
            end = pd.Timestamp(end_date).normalize()
            current = start.replace(day=1)
            while current <= end:
                next_month = current + pd.offsets.MonthBegin(1)
                yield max(start, current), min(end, next_month - pd.Timedelta(days=1))
                current = next_month

        def _sum_double(columns):
            terms = " + ".join(
                f"COALESCE(CAST({column} AS DOUBLE), 0.0)"
                for column in columns
            )
            return f"CAST(({terms}) AS DOUBLE)"

        def _weighted_sum(columns, weights):
            terms = " + ".join(
                f"COALESCE(CAST({column} AS DOUBLE), 0.0) "
                f"* CAST({float(weight):.10f} AS DOUBLE)"
                for column, weight in zip(columns, weights)
            )
            return f"CAST(({terms}) AS DOUBLE)"

        def _book_level_ofi_expr(level: int):
            bid = f"""
                CASE
                    WHEN bid_p{level} <= 0 OR prev_bid_p{level} <= 0 THEN NULL
                    WHEN bid_p{level} > prev_bid_p{level} THEN bid_v{level}
                    WHEN bid_p{level} = prev_bid_p{level}
                        THEN bid_v{level} - prev_bid_v{level}
                    ELSE -prev_bid_v{level}
                END
            """
            ask = f"""
                CASE
                    WHEN ask_p{level} <= 0 OR prev_ask_p{level} <= 0 THEN NULL
                    WHEN ask_p{level} < prev_ask_p{level} THEN ask_v{level}
                    WHEN ask_p{level} = prev_ask_p{level}
                        THEN ask_v{level} - prev_ask_v{level}
                    ELSE -prev_ask_v{level}
                END
            """
            return bid, ask

        def build_mlofi_daily_sql(
            bar_table: str,
            month_start: pd.Timestamp,
            month_end: pd.Timestamp,
        ) -> str:
            levels = list(BOOK_LEVELS)
            weights = [math.exp(-0.35 * (level - 1)) for level in levels]

            raw_level_columns = []
            lag_level_columns = []
            level_event_columns = []
            level_ofi_columns = []
            level_norm_columns = []
            level_daily_aggregates = []

            for level in levels:
                raw_level_columns.extend([
                    f"CAST(bid_price{level} AS DOUBLE) AS bid_p{level}",
                    f"CAST(ask_price{level} AS DOUBLE) AS ask_p{level}",
                    f"COALESCE(CAST(bid_volume{level} AS DOUBLE), 0.0) AS bid_v{level}",
                    f"COALESCE(CAST(ask_volume{level} AS DOUBLE), 0.0) AS ask_v{level}",
                ])
                lag_level_columns.extend([
                    f"LAG(bid_p{level}, 1) OVER "
                    f"(PARTITION BY instrument, trading_day ORDER BY date) "
                    f"AS prev_bid_p{level}",
                    f"LAG(ask_p{level}, 1) OVER "
                    f"(PARTITION BY instrument, trading_day ORDER BY date) "
                    f"AS prev_ask_p{level}",
                    f"LAG(bid_v{level}, 1) OVER "
                    f"(PARTITION BY instrument, trading_day ORDER BY date) "
                    f"AS prev_bid_v{level}",
                    f"LAG(ask_v{level}, 1) OVER "
                    f"(PARTITION BY instrument, trading_day ORDER BY date) "
                    f"AS prev_ask_v{level}",
                ])
                bid_expr, ask_expr = _book_level_ofi_expr(level)
                level_event_columns.extend([
                    f"{bid_expr} AS bid_event_{level}",
                    f"{ask_expr} AS ask_event_{level}",
                ])
                level_ofi_columns.append(
                    f"bid_event_{level} - ask_event_{level} AS ofi_level_{level}"
                )
                level_norm_columns.append(
                    f"ofi_level_{level} / "
                    f"(0.5 * (total_depth + prev_total_depth) + {EPS}) "
                    f"AS nofi_level_{level}"
                )
                level_daily_aggregates.extend([
                    f"SUM(nofi_level_{level}) AS nofi_level_{level}__sum",
                    f"AVG(ABS(nofi_level_{level})) AS nofi_level_{level}__abs_mean",
                ])

            raw_level_sql = ",\n            ".join(raw_level_columns)
            lag_level_sql = ",\n            ".join(lag_level_columns)
            event_sql = ",\n            ".join(level_event_columns)
            ofi_sql = ",\n            ".join(level_ofi_columns)
            norm_sql = ",\n            ".join(level_norm_columns)
            daily_level_sql = ",\n            ".join(level_daily_aggregates)

            bid_volume_columns = [f"bid_v{level}" for level in levels]
            ask_volume_columns = [f"ask_v{level}" for level in levels]
            total_depth_expr = (
                f"{_sum_double(bid_volume_columns)} + "
                f"{_sum_double(ask_volume_columns)}"
            )
            weighted_bid_expr = _weighted_sum(bid_volume_columns, weights)
            weighted_ask_expr = _weighted_sum(ask_volume_columns, weights)

            weighted_ofi_terms = " + ".join(
                f"COALESCE(ofi_level_{level}, 0.0) "
                f"* CAST({weight:.10f} AS DOUBLE)"
                for level, weight in zip(levels, weights)
            )
            weight_sum = float(sum(weights))

            near_levels = levels[:2]
            far_levels = levels[2:]
            near_ofi_expr = " + ".join(
                f"COALESCE(ofi_level_{level}, 0.0)" for level in near_levels
            )
            far_ofi_expr = " + ".join(
                f"COALESCE(ofi_level_{level}, 0.0)" for level in far_levels
            )

            bid_centroid_num = " + ".join(
                f"CAST({level} AS DOUBLE) * bid_v{level}" for level in levels
            )
            ask_centroid_num = " + ".join(
                f"CAST({level} AS DOUBLE) * ask_v{level}" for level in levels
            )
            bid_depth_expr = _sum_double(bid_volume_columns)
            ask_depth_expr = _sum_double(ask_volume_columns)

            month_start_literal = _timestamp_string(month_start)
            month_end_literal = _timestamp_string(
                pd.Timestamp(month_end).normalize()
                + pd.Timedelta(hours=23, minutes=59, seconds=59)
            )

            return f"""
            WITH universe AS (
                SELECT
                    CAST(date AS DATE) AS trading_day,
                    instrument
                FROM {INSTRUMENT_TABLE}
                WHERE date >= TIMESTAMP '{month_start_literal}'
                  AND date <= TIMESTAMP '{month_end_literal}'
            ),

            raw AS (
                SELECT
                    b.date,
                    b.instrument,
                    CAST(b.date AS DATE) AS trading_day,
                    CAST(b.close AS DOUBLE) AS close_float,
                    {raw_level_sql}
                FROM {bar_table} b
                PRUNE JOIN universe u
                  ON CAST(b.date AS DATE) = u.trading_day
                 AND b.instrument = u.instrument
                WHERE b.date >= TIMESTAMP '{month_start_literal}'
                  AND b.date <= TIMESTAMP '{month_end_literal}'
                  AND b.bid_price1 > 0
                  AND b.ask_price1 > 0
                  AND b.ask_price1 >= b.bid_price1
            ),

            book AS (
                SELECT
                    *,
                    (ask_p1 + bid_p1) / 2.0 AS mid_price,
                    (ask_p1 - bid_p1)
                        / (((ask_p1 + bid_p1) / 2.0) + {EPS}) * 10000.0
                        AS spread_bps,
                    {total_depth_expr} AS total_depth,
                    {bid_depth_expr} AS bid_depth,
                    {ask_depth_expr} AS ask_depth,
                    {weighted_bid_expr} AS weighted_bid_depth,
                    {weighted_ask_expr} AS weighted_ask_depth,
                    ({bid_centroid_num}) / ({bid_depth_expr} + {EPS})
                        AS bid_depth_centroid,
                    ({ask_centroid_num}) / ({ask_depth_expr} + {EPS})
                        AS ask_depth_centroid,
                    (bid_p1 - bid_p{levels[-1]})
                        / (((ask_p1 + bid_p1) / 2.0) + {EPS}) * 10000.0
                        AS bid_gap_bps,
                    (ask_p{levels[-1]} - ask_p1)
                        / (((ask_p1 + bid_p1) / 2.0) + {EPS}) * 10000.0
                        AS ask_gap_bps
                FROM raw
            ),

            lagged AS (
                SELECT
                    *,
                    LAG(mid_price, 1) OVER (
                        PARTITION BY instrument, trading_day ORDER BY date
                    ) AS prev_mid_price,
                    LAG(total_depth, 1) OVER (
                        PARTITION BY instrument, trading_day ORDER BY date
                    ) AS prev_total_depth,
                    LEAD(mid_price, 5) OVER (
                        PARTITION BY instrument, trading_day ORDER BY date
                    ) AS future_mid_5,
                    LEAD(spread_bps, 5) OVER (
                        PARTITION BY instrument, trading_day ORDER BY date
                    ) AS future_spread_5,
                    LEAD(total_depth, 5) OVER (
                        PARTITION BY instrument, trading_day ORDER BY date
                    ) AS future_depth_5,
                    {lag_level_sql}
                FROM book
            ),

            level_events AS (
                SELECT
                    *,
                    {event_sql}
                FROM lagged
            ),

            level_flows AS (
                SELECT
                    *,
                    {ofi_sql}
                FROM level_events
            ),

            raw_flow AS (
                SELECT
                    *,
                    ({weighted_ofi_terms}) / CAST({weight_sum:.10f} AS DOUBLE)
                        AS integrated_ofi_raw,
                    ({near_ofi_expr}) AS near_ofi_raw,
                    ({far_ofi_expr}) AS far_ofi_raw
                FROM level_flows
            ),

            normalized_flow AS (
                SELECT
                    *,
                    {norm_sql},
                    integrated_ofi_raw
                        / (0.5 * (total_depth + prev_total_depth) + {EPS})
                        AS mlofi,
                    near_ofi_raw
                        / (0.5 * (total_depth + prev_total_depth) + {EPS})
                        AS near_ofi,
                    far_ofi_raw
                        / (0.5 * (total_depth + prev_total_depth) + {EPS})
                        AS far_ofi,
                    CASE
                        WHEN mid_price > 0 AND prev_mid_price > 0
                        THEN LOG(mid_price / prev_mid_price) * 10000.0
                        ELSE NULL
                    END AS mid_return_bps,
                    bid_depth_centroid - ask_depth_centroid
                        AS depth_centroid_asymmetry,
                    (bid_gap_bps - ask_gap_bps)
                        AS price_gap_asymmetry,
                    (bid_v1 + ask_v1) / (total_depth + {EPS})
                        AS top_depth_concentration
                FROM raw_flow
            ),

            ranked_flow AS (
                SELECT
                    *,
                    LAG(mlofi, 1) OVER (
                        PARTITION BY instrument, trading_day ORDER BY date
                    ) AS prev_mlofi,
                    PERCENT_RANK() OVER (
                        PARTITION BY instrument, trading_day
                        ORDER BY ABS(mlofi)
                    ) AS abs_mlofi_percentile
                FROM normalized_flow
            ),

            dynamics AS (
                SELECT
                    *,
                    CASE
                        WHEN abs_mlofi_percentile >= {SHOCK_PERCENTILE}
                        THEN 1 ELSE 0
                    END AS shock_flag,
                    CASE WHEN mlofi > 0 THEN 1 WHEN mlofi < 0 THEN -1 ELSE 0 END
                        AS flow_sign,
                    CASE
                        WHEN prev_mlofi IS NULL OR prev_mlofi = 0 OR mlofi = 0
                        THEN NULL
                        WHEN prev_mlofi * mlofi > 0 THEN 1.0
                        ELSE 0.0
                    END AS same_sign_flow,
                    CASE
                        WHEN spread_bps > 0 AND future_spread_5 IS NOT NULL
                        THEN (spread_bps - future_spread_5)
                            / (spread_bps + {EPS})
                        ELSE NULL
                    END AS spread_recovery_5,
                    CASE
                        WHEN total_depth >= 0 AND future_depth_5 IS NOT NULL
                        THEN LOG((future_depth_5 + 1.0) / (total_depth + 1.0))
                        ELSE NULL
                    END AS depth_recovery_5,
                    CASE
                        WHEN mid_price > 0 AND future_mid_5 > 0
                        THEN
                            (CASE WHEN mlofi > 0 THEN 1.0
                                  WHEN mlofi < 0 THEN -1.0
                                  ELSE 0.0 END)
                            * LOG(future_mid_5 / mid_price) * 10000.0
                        ELSE NULL
                    END AS continuation_5,
                    CASE
                        WHEN future_spread_5 IS NULL OR future_depth_5 IS NULL
                        THEN NULL
                        WHEN future_spread_5 >= spread_bps
                         AND future_depth_5 <= total_depth
                        THEN 1.0 ELSE 0.0
                    END AS non_recovery_5
                FROM ranked_flow
            ),

            daily_raw AS (
                SELECT
                    CAST(trading_day AS DATETIME) AS date,
                    instrument,
                    COUNT(*) AS minute_count,
                    LAST(close_float ORDER BY date) AS daily_close,

                    {daily_level_sql},

                    SUM(mlofi) AS mlofi__sum,
                    AVG(mlofi) AS mlofi__mean,
                    NANSTD(mlofi) AS mlofi__std,
                    AVG(ABS(mlofi)) AS mlofi__abs_mean,
                    LAST(mlofi ORDER BY date) AS mlofi__last,
                    AVG(CASE
                        WHEN STRFTIME(date, '%H:%M:%S') >= '14:30:00'
                        THEN mlofi ELSE NULL END
                    ) AS mlofi__close30_mean,

                    SUM(near_ofi) AS near_ofi__sum,
                    SUM(far_ofi) AS far_ofi__sum,
                    SUM(near_ofi - far_ofi) AS near_minus_far_ofi__sum,

                    AVG(same_sign_flow) AS mlofi_persistence__mean,
                    AVG(CASE
                        WHEN same_sign_flow IS NULL THEN NULL
                        ELSE 1.0 - same_sign_flow END
                    ) AS mlofi_flip_rate,

                    AVG(CAST(shock_flag AS DOUBLE)) AS shock_rate,
                    AVG(CASE
                        WHEN flow_sign > 0 THEN CAST(shock_flag AS DOUBLE)
                        ELSE 0.0 END
                    ) AS buy_shock_rate,
                    AVG(CASE
                        WHEN flow_sign < 0 THEN CAST(shock_flag AS DOUBLE)
                        ELSE 0.0 END
                    ) AS sell_shock_rate,
                    AVG(CASE
                        WHEN STRFTIME(date, '%H:%M:%S') >= '14:30:00'
                        THEN CAST(shock_flag AS DOUBLE)
                        ELSE NULL END
                    ) AS close30_shock_rate,

                    AVG(CASE WHEN shock_flag = 1
                        THEN spread_recovery_5 ELSE NULL END
                    ) AS shock_spread_recovery_5__mean,
                    AVG(CASE WHEN shock_flag = 1
                        THEN depth_recovery_5 ELSE NULL END
                    ) AS shock_depth_recovery_5__mean,
                    AVG(CASE WHEN shock_flag = 1
                        THEN continuation_5 ELSE NULL END
                    ) AS shock_continuation_5__mean,
                    AVG(CASE WHEN shock_flag = 1
                        THEN -continuation_5 ELSE NULL END
                    ) AS shock_reversal_5__mean,
                    AVG(CASE WHEN shock_flag = 1
                        THEN non_recovery_5 ELSE NULL END
                    ) AS shock_non_recovery_5__rate,

                    AVG(CASE WHEN shock_flag = 1 AND flow_sign > 0
                        THEN continuation_5 ELSE NULL END
                    ) AS buy_shock_continuation_5__mean,
                    AVG(CASE WHEN shock_flag = 1 AND flow_sign < 0
                        THEN continuation_5 ELSE NULL END
                    ) AS sell_shock_continuation_5__mean,

                    AVG(bid_gap_bps) AS bid_gap_bps__mean,
                    NANSTD(bid_gap_bps) AS bid_gap_bps__std,
                    AVG(ask_gap_bps) AS ask_gap_bps__mean,
                    NANSTD(ask_gap_bps) AS ask_gap_bps__std,
                    AVG(price_gap_asymmetry) AS price_gap_asymmetry__mean,
                    NANSTD(price_gap_asymmetry) AS price_gap_asymmetry__std,
                    AVG(depth_centroid_asymmetry)
                        AS depth_centroid_asymmetry__mean,
                    NANSTD(depth_centroid_asymmetry)
                        AS depth_centroid_asymmetry__std,
                    AVG(top_depth_concentration)
                        AS top_depth_concentration__mean,
                    NANSTD(top_depth_concentration)
                        AS top_depth_concentration__std,

                    (
                        AVG(mlofi * mid_return_bps)
                        - AVG(mlofi) * AVG(mid_return_bps)
                    ) / (
                        AVG(mlofi * mlofi)
                        - AVG(mlofi) * AVG(mlofi)
                        + {EPS}
                    ) AS impact_lambda,

                    (
                        (
                            AVG(mlofi * mid_return_bps)
                            - AVG(mlofi) * AVG(mid_return_bps)
                        )
                        *
                        (
                            AVG(mlofi * mid_return_bps)
                            - AVG(mlofi) * AVG(mid_return_bps)
                        )
                    ) / (
                        (
                            AVG(mlofi * mlofi)
                            - AVG(mlofi) * AVG(mlofi)
                            + {EPS}
                        )
                        *
                        (
                            AVG(mid_return_bps * mid_return_bps)
                            - AVG(mid_return_bps) * AVG(mid_return_bps)
                            + {EPS}
                        )
                    ) AS impact_r2_proxy
                FROM dynamics
                GROUP BY instrument, trading_day
            )

            SELECT
                *,
                mlofi__close30_mean - mlofi__mean
                    AS mlofi__close30_minus_full,
                impact_lambda * mlofi__sum
                    AS lambda_x_mlofi_sum,
                shock_spread_recovery_5__mean
                    + shock_depth_recovery_5__mean
                    - shock_continuation_5__mean / 10.0
                    AS resiliency_composite,
                mlofi__sum * (1.0 - COALESCE(shock_non_recovery_5__rate, 0.0))
                    AS absorbed_flow,
                mlofi__sum * COALESCE(shock_non_recovery_5__rate, 0.0)
                    AS unabsorbed_flow
            FROM daily_raw
            WHERE minute_count >= {MIN_VALID_MINUTES}
            ORDER BY date, instrument
            """

        query_start = _as_timestamp(block_start).normalize()
        query_end = _as_timestamp(block_end).normalize()
        pieces = []
        for month_start, month_end in _month_intervals(query_start, query_end):
            sql = build_mlofi_daily_sql(bar_table, month_start, month_end)
            part = dai.query(sql, compression=True).df()
            if part.empty:
                continue
            part["date"] = pd.to_datetime(part["date"]).dt.normalize()
            part["instrument"] = part["instrument"].astype(str)
            pieces.append(part)
        if not pieces:
            raise RuntimeError("Phase-3 feature query returned no rows.")
        frame = pd.concat(pieces, ignore_index=True)
        missing = [column for column in PHASE3_FEATURES if column not in frame.columns]
        if missing:
            raise KeyError(f"Phase-3 block missing columns: {missing[:20]}")
        return (
            frame.loc[
                frame["date"].between(query_start, query_end),
                ["date", "instrument", *PHASE3_FEATURES],
            ]
            .drop_duplicates(["date", "instrument"], keep="last")
            .sort_values(["date", "instrument"])
            .reset_index(drop=True)
        )

    def _build_unified_blocks(block_datasources, block_start, block_end, training):
        zoo = _build_e4_features(block_datasources, block_start, block_end)
        phase1 = _build_phase1_features(block_datasources["bar1m"], block_start, block_end)
        phase3 = _build_phase3_features(block_datasources["bar1m"], block_start, block_end)

        zoo_z = _cs_z(zoo, E4_FEATURES).rename(columns={column: f"zoo__{column}" for column in E4_FEATURES})
        phase1_z = _cs_z(phase1, PHASE1_FEATURES).rename(columns={column: f"p1__{column}" for column in PHASE1_FEATURES})
        phase3_z = _cs_z(phase3, PHASE3_FEATURES).rename(columns={column: f"p3__{column}" for column in PHASE3_FEATURES})

        if training:
            base = zoo[["date", "instrument"]].copy()
        else:
            pool = _query_pool(block_start, block_end)
            zoo_keys = zoo[["date", "instrument"]].drop_duplicates()
            missing_dates = set(pool["date"].unique()) - set(zoo_keys["date"].unique())
            fallback = pd.concat(
                [phase1[["date", "instrument"]], phase3[["date", "instrument"]]],
                ignore_index=True,
            ).drop_duplicates()
            fallback = fallback[fallback["date"].isin(missing_dates)]
            base = pd.concat([zoo_keys, fallback], ignore_index=True).drop_duplicates()
            base = pool.merge(base, on=["date", "instrument"], how="inner", validate="one_to_one")

        panel = base.merge(zoo_z, on=["date", "instrument"], how="left", validate="one_to_one")
        panel = panel.merge(phase1_z, on=["date", "instrument"], how="left", validate="one_to_one")
        panel = panel.merge(phase3_z, on=["date", "instrument"], how="left", validate="one_to_one")
        for column in ALL_FEATURES:
            panel[column] = pd.to_numeric(panel[column], errors="coerce").astype(np.float32)
        return panel.sort_values(["date", "instrument"]).reset_index(drop=True)

    public_sources = {"bar1m": TRAIN_BAR1M_TABLE, "financial": TRAIN_FINANCIAL_TABLE}
    training = _build_unified_blocks(public_sources, TRAIN_START, TRAIN_END, training=True)
    daily_close = _query_daily_close(
        TRAIN_BAR1M_TABLE,
        TRAIN_START,
        TRAIN_END + pd.Timedelta(days=LABEL_BUFFER_DAYS),
    )
    labels = _attach_raw_forward_return(daily_close)
    training = training.merge(labels, on=["date", "instrument"], how="left", validate="one_to_one")
    training = training[training["date"].between(TRAIN_START, TRAIN_END) & training["target_return"].notna()].reset_index(drop=True)
    if training.empty:
        raise RuntimeError("Fixed public training sample is empty.")

    development = _sample_per_date(training[training["date"].le(DEV_END)], MAX_ROWS_PER_DATE)
    validation = _sample_per_date(training[training["date"].gt(DEV_END)], MAX_ROWS_PER_DATE)
    refit = _sample_per_date(training, MAX_ROWS_PER_DATE)
    if development.empty or validation.empty or refit.empty:
        raise RuntimeError("Development/validation/refit sample is empty.")

    parameters = dict(
        objective="reg:squarederror",
        eval_metric="rmse",
        n_estimators=700,
        max_depth=3,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=50.0,
        reg_alpha=0.1,
        reg_lambda=10.0,
        tree_method="hist",
        n_jobs=THREADS,
        random_state=RANDOM_STATE,
    )
    probe = xgb.XGBRegressor(**parameters)
    probe.fit(
        development[ALL_FEATURES].fillna(0.0).to_numpy(np.float32),
        development["target_return"].to_numpy(np.float32),
        eval_set=[(
            validation[ALL_FEATURES].fillna(0.0).to_numpy(np.float32),
            validation["target_return"].to_numpy(np.float32),
        )],
        early_stopping_rounds=60,
        verbose=False,
    )
    best_trees = int(getattr(probe, "best_iteration", 699)) + 1
    parameters["n_estimators"] = best_trees
    model = xgb.XGBRegressor(**parameters)
    model.fit(
        refit[ALL_FEATURES].fillna(0.0).to_numpy(np.float32),
        refit["target_return"].to_numpy(np.float32),
        verbose=False,
    )

    del training, development, validation, refit, daily_close, labels, probe
    gc.collect()

    evaluation_start = _as_timestamp(start_date).normalize()
    evaluation_end = _as_timestamp(end_date).normalize()
    if evaluation_end < evaluation_start:
        raise ValueError("end_date is earlier than start_date.")

    test = _build_unified_blocks(datasources, evaluation_start, evaluation_end, training=False)
    if test.empty:
        raise RuntimeError("Injected test panel is empty.")
    raw_score = model.predict(test[ALL_FEATURES].fillna(0.0).to_numpy(np.float32))
    result = test[["date", "instrument"]].copy()
    result["_raw_score"] = np.asarray(raw_score, dtype=float)
    result["factor"] = result.groupby("date", sort=False)["_raw_score"].rank(method="average", pct=True) - 0.5
    result = result[["date", "instrument", "factor"]]
    result["date"] = pd.to_datetime(result["date"]).dt.normalize()
    result["instrument"] = result["instrument"].astype(str)
    result["factor"] = pd.to_numeric(result["factor"], errors="coerce").replace([np.inf, -np.inf], np.nan)
    result = result.dropna(subset=["factor"]).drop_duplicates(["date", "instrument"], keep="last").sort_values(["date", "instrument"]).reset_index(drop=True)

    pool = _query_pool(evaluation_start, evaluation_end)
    expected_dates = set(pool["date"].unique())
    output_dates = set(result["date"].unique())
    if result.empty:
        raise RuntimeError("main() produced an empty DataFrame.")
    if list(result.columns) != ["date", "instrument", "factor"]:
        raise RuntimeError("Output columns must be exactly date/instrument/factor.")
    if result.duplicated(["date", "instrument"]).any():
        raise RuntimeError("Output contains duplicate date/instrument rows.")
    if not np.isfinite(result["factor"].to_numpy(float)).all():
        raise RuntimeError("Output factor contains non-finite values.")
    if output_dates != expected_dates:
        raise RuntimeError(f"Trading-day mismatch: missing={sorted(expected_dates-output_dates)[:5]}, extra={sorted(output_dates-expected_dates)[:5]}")

    coverage = pool.groupby("date").size().rename("pool_rows").to_frame().join(
        result.groupby("date").size().rename("factor_rows"), how="left"
    ).fillna(0)
    coverage["coverage"] = coverage["factor_rows"] / coverage["pool_rows"]
    if float(coverage["coverage"].min()) < 0.60:
        raise RuntimeError(f"Daily coverage below 60%: {coverage['coverage'].nsmallest(5).to_dict()}")
    daily_unique = result.groupby("date")["factor"].nunique(dropna=True)
    daily_std = result.groupby("date")["factor"].std(ddof=0)
    bad = daily_unique.le(1) | daily_std.fillna(0).le(1e-12)
    if bad.any():
        raise RuntimeError(f"Factor degenerates on dates: {[str(value.date()) for value in bad[bad].index[:5]]}")

    return result


## Disabled local smoke test

In [ ]:
RUN_LOCAL_TEST = False

if RUN_LOCAL_TEST:
    test_datasources = {
        "bar1m": "bigalpha_2026_stock_bar1m",
        "financial": "bigalpha_2026_financial",
    }
    smoke = main(
        test_datasources,
        "2024-01-01 00:00:00",
        "2024-01-31 23:59:59",
    )
    print(smoke.shape)
    print(smoke.groupby("date")["factor"].agg(["count", "nunique", "std"]).tail())
    print(smoke.tail())
